In [1]:
import traceback
try:
    exec(compile('import os, glob\n# Kaggle has used both mounts; resolve at runtime and expose a stable path.\n_cands = [d for d in ["/kaggle/input/competitions/arc-prize-2026-arc-agi-2", "/kaggle/input/arc-prize-2026-arc-agi-2"] if os.path.exists(d + "/arc-agi_test_challenges.json")]\n_cands += [os.path.dirname(f) for f in glob.glob("/kaggle/input/**/arc-agi_test_challenges.json", recursive=True)]\nassert _cands, "competition data not found under /kaggle/input"\nos.makedirs("/kaggle/working", exist_ok=True)\nif not os.path.exists("/kaggle/working/arcdata"):\n    os.symlink(_cands[0], "/kaggle/working/arcdata")\nprint("ARC data:", _cands[0])\nimport time\n_T0 = time.time()\nQWEN_DEADLINE = _T0 + 12 * 3600 - 2400      # whole run must end 40 min before the 12h limit\nglobal_end_time = _T0 + 6.5 * 3600  # NVARC phase ends here; Qwen gets the rest\nQWEN_EFFORT = \'\'; QWEN_MAX_TOKENS = 64000; QWEN_COMMIT_TASKS = 24; QWEN_COMMIT_MIN = 45; QWEN_WORKERS = 24; QWEN_TPS = 15.0; QWEN_EXTRA_TRIES = [[\'--language-model-only\', \'--enable-prefix-caching\', \'--reasoning-config\', \'{"reasoning_start_str": "<think>", "reasoning_end_str": "\\\\n\\\\nI am out of thinking budget. I will now commit to the most likely rule found above, write it as the Python function `transform`, then write the final output grid(s).\\\\n</think>\\\\n\\\\n"}\'], [\'--language-model-only\', \'--enable-prefix-caching\'], [\'--language-model-only\']]; QWEN_GLOB = \'qwen3-8-27b*\'; QWEN_MERGE_MAXCOUNT = 3; QWEN_STOP_OVERRIDE = True\nos.environ[\'QWEN_FORCE\'] = \'1\'\nos.environ[\'QWEN_PROGRAM\'] = \'1\'\nos.environ[\'QWEN_V14\'] = \'1\'\nos.environ[\'QWEN_REPAIR\'] = \'after\'; os.environ[\'QWEN_REPAIR_TOKENS\'] = \'24000\'\nos.environ[\'QWEN_THINK_BUDGET\'] = \'6000\'\nimport json as _j, os as _o\n_p = "/kaggle/working/arcdata/" + ("arc-agi_test_challenges.json" if _o.getenv("KAGGLE_IS_COMPETITION_RERUN") else "arc-agi_evaluation_challenges.json")\n_j.dump({k: [{"attempt_1": [[0]], "attempt_2": [[0]]} for _ in v["test"]] for k, v in _j.load(open(_p)).items()}, open("submission.json", "w"))\nprint("fallback submission written")\n', 'added_cell', 'exec'), globals())
except BaseException as _e:
    print('ADDED CELL FAILED (NVARC submission kept):', repr(_e)); traceback.print_exc()


ARC data: /kaggle/input/competitions/arc-prize-2026-arc-agi-2
fallback submission written


In [2]:
import traceback
try:
    exec(compile('# === Qwen env: standalone CPython 3.12 + offline vLLM wheelhouse, built in the background ===\nimport os, glob, subprocess, time\ndef _one(pattern, what):\n    hits = sorted(glob.glob(pattern, recursive=True))\n    print(what, "->", hits[:1]); return hits[0] if hits else None\nPY_TAR = _one("/kaggle/input/**/cpython312_install_only.tgzbin", "py312 tarball")  # odd extension: Kaggle auto-extracts .tar.gz\nWHEELS = os.path.dirname(_one("/kaggle/input/**/requirements.lock", "wheelhouse") or "/none/x")\nQWEN_MODEL = os.path.dirname(_one(f"/kaggle/input/**/{QWEN_GLOB}/**/config.json", "qwen model") or _one(f"/kaggle/input/**/{QWEN_GLOB}/config.json", "qwen model") or "/none/x")\nVENV = "/kaggle/tmp/vllmenv"; VPY = VENV + "/bin/python"\nsetup_sh = f"""set -x\nunset PYTHONPATH PYTHONHOME PYTHONSTARTUP; export PYTHONNOUSERSITE=1  # NVARC env leaked into the venv (pip skipped typing_extensions etc.)\nmkdir -p /kaggle/tmp/py312 && tar -xzf \'{PY_TAR}\' -C /kaggle/tmp/py312\n/kaggle/tmp/py312/python/bin/python3.12 -m venv {VENV}\n{VENV}/bin/pip install -q --no-index --find-links \'{WHEELS}\' -r \'{WHEELS}/requirements.lock\'\n{VENV}/bin/pip install -q --no-index --find-links \'{WHEELS}\' ninja || true\nfor f in $(ls {VENV}/lib/python3.12/site-packages/vllm/model_executor/layers/attention/attention.py 2>/dev/null); do\n  grep -q "extra_impl_args.pop(\'layer_idx\', None)" $f || sed -i "s/self.impl = impl_cls(/extra_impl_args.pop(\'layer_idx\', None)\\\\n        self.impl = impl_cls(/" $f\ndone\n{VPY} -c \'import vllm, torch; print("VLLM", vllm.__version__, torch.__version__)\' && echo VENV_DONE || echo VENV_FAILED\n"""\nopen("/kaggle/working/venv_setup.sh", "w").write(setup_sh)\nif PY_TAR and WHEELS != "/none" and QWEN_MODEL != "/none":\n    subprocess.Popen("nohup bash /kaggle/working/venv_setup.sh > /kaggle/working/venv_setup.log 2>&1 &",\n                     shell=True, start_new_session=True)\n    print("venv setup started in background")\nelse:\n    print("Qwen inputs missing -> hybrid disabled, NVARC only", PY_TAR, WHEELS, QWEN_MODEL)\n', 'added_cell', 'exec'), globals())
except BaseException as _e:
    print('ADDED CELL FAILED (NVARC submission kept):', repr(_e)); traceback.print_exc()


py312 tarball -> ['/kaggle/input/datasets/yukuo14/py312-standalone-linux/cpython312_install_only.tgzbin']
wheelhouse -> ['/kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3/requirements.lock']
qwen model -> ['/kaggle/input/models/mikedan7/qwen3-8-27b-fp8-official/pytorch/hf-fp8/1/config.json']
venv setup started in background


In [3]:
%%writefile /kaggle/working/qwen_solver.py
"""Qwen3.8 program-synthesis solver for ARC-AGI-2 (hybrid with the NVARC pipeline).

For each task the LLM writes `transform(grid) -> grid` in Python. The program is executed in a
sandboxed subprocess on every training pair; only programs that reproduce ALL training outputs are
trusted ("verified"). Failed attempts are retried with feedback (which pairs failed, and how).

Output (one JSON per task, written as soon as the task finishes, so a killed run keeps its work):
  {"task": id, "verified": bool, "tests": [grid_or_null, ...], "rounds": n, "seconds": s}

merge_with_nvarc(): verified program output becomes attempt_1, NVARC's best becomes attempt_2;
unverified tasks keep NVARC's two attempts untouched (so this can never lower a task's score
except when a verified program is wrong AND NVARC's #2 was the right one).

Usage (inside the 12h notebook, after the vLLM server is up):
  python qwen_solver.py --tasks <challenges.json> --out <dir> --deadline-ts <unix> [--workers 16]
"""
import argparse
import json
import os
import re
import subprocess
import sys
import tempfile
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor

import threading
_last = threading.local()
API = os.environ.get("QWEN_API", "http://127.0.0.1:8000/v1/chat/completions")
MODEL = os.environ.get("QWEN_MODEL", "qwen3.8-27b")

PROMPT = """You are solving an ARC-AGI puzzle. Each grid is a list of rows; cells are colors 0-9.
Find the single transformation rule that maps every training input to its output, then write it
as a Python function.

{pairs}

Test input(s):
{tests}

Write a complete, self-contained Python 3 function:

```python
def transform(grid):
    # grid: list[list[int]]; return list[list[int]]
    ...
```

Rules: standard library only (you may import copy, itertools, collections, math); no I/O; it must
work on every training input above AND generalize to the test input(s). Reply with the final code
in one ```python block."""

FEEDBACK = """Your program does not reproduce every training pair:
{report}
Re-examine the examples, fix the rule, and reply with the complete corrected function in one
```python block."""

RUNNER = r'''
import json, sys
try:
    import resource
    resource.setrlimit(resource.RLIMIT_AS, (2 << 30, 2 << 30))
except Exception:
    pass
src, grids = json.load(sys.stdin)
ns = {}
out, err = [], []
try:
    exec(src, ns)
    fn = ns["transform"]
except Exception as e:
    fn, bad = None, f"{type(e).__name__}: {e}"[:300]
for g in grids:
    if fn is None:
        out.append(None); err.append(bad); continue
    try:
        r = fn([list(row) for row in g])
        if hasattr(r, "tolist"):  # numpy array
            r = r.tolist()
        r = [[int(v) for v in row] for row in r]
        ok = len(r) > 0 and all(len(row) == len(r[0]) and 0 < len(row) <= 30 and all(0 <= v <= 9 for v in row)
                                for row in r) and len(r) <= 30
        out.append(r if ok else None)
        err.append(None if ok else "returned an invalid grid (must be 1-30 rows of equal length, values 0-9)")
    except Exception as e:
        out.append(None)
        err.append(f"{type(e).__name__}: {e}"[:300])
print(json.dumps({"out": out, "err": err}))
'''


def grid_text(g):
    return "\n".join("".join(str(c) for c in row) for row in g)


def build_prompt(task):
    pairs = "\n\n".join(f"Example {i+1} input:\n{grid_text(p['input'])}\nExample {i+1} output:\n{grid_text(p['output'])}"
                        for i, p in enumerate(task["train"]))
    tests = "\n\n".join(f"Test {i+1} input:\n{grid_text(t['input'])}" for i, t in enumerate(task["test"]))
    return PROMPT.format(pairs=pairs, tests=tests)


def extract_code(text):
    blocks = re.findall(r"```(?:python)?\s*\n(.*?)```", text or "", re.S)
    blocks = [b for b in blocks if "def transform" in b]
    return blocks[-1] if blocks else None


def run_program_err(code, grids, timeout=10):
    """(outputs, error messages) of transform() on each grid; None output = crash, invalid grid or timeout."""
    env = dict(os.environ, OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")  # numpy under RLIMIT_AS
    try:
        p = subprocess.run([sys.executable, "-I", "-c", RUNNER], input=json.dumps([code, grids]),
                           capture_output=True, text=True, timeout=timeout, env=env)
        r = json.loads(p.stdout.strip().splitlines()[-1])
        return r["out"], r["err"]
    except subprocess.TimeoutExpired:
        return [None] * len(grids), [f"timed out after {timeout}s"] * len(grids)
    except Exception as e:
        return [None] * len(grids), [f"runner failed: {e!r}"[:200]] * len(grids)


def run_program(code, grids, timeout=10):
    return run_program_err(code, grids, timeout)[0]


def diff_report(task, outs, errs=None):
    lines = []
    for i, (p, o) in enumerate(zip(task["train"], outs)):
        exp = p["output"]
        if o == exp:
            lines.append(f"- Example {i+1}: correct")
        elif o is None:
            lines.append(f"- Example {i+1}: " + ((errs[i] if errs and errs[i] else None) or "crashed or returned an invalid grid"))
        elif len(o) != len(exp) or len(o[0]) != len(exp[0]):
            lines.append(f"- Example {i+1}: wrong shape {len(o)}x{len(o[0])} (expected {len(exp)}x{len(exp[0])})")
        else:
            bad = sum(a != b for ra, rb in zip(o, exp) for a, b in zip(ra, rb))
            lines.append(f"- Example {i+1}: {bad} cells wrong. Your output:\n{grid_text(o)}")
    return "\n".join(lines)


def chat(messages, deadline_ts, max_tokens=32000, temperature=0.6):
    remaining = deadline_ts - time.time()
    if remaining < 30:
        raise TimeoutError("deadline")
    body = {"model": MODEL, "messages": messages, "max_tokens": max_tokens, "temperature": temperature,
            "chat_template_kwargs": dict({"enable_thinking": True}, **({"reasoning_effort": os.environ["QWEN_EFFORT"]} if os.environ.get("QWEN_EFFORT") else {}))}
    if V14:  # exact token ids, so a forced answer can extend this very sequence and hit the prefix cache
        body["return_token_ids"] = True
    reserve = int(os.environ.get("QWEN_THINK_BUDGET", "0") or 0)
    if reserve and max_tokens > reserve + 2000:  # v15: vLLM closes the think block itself (--reasoning-config), leaving
        body["thinking_token_budget"] = max_tokens - reserve  # `reserve` tokens for the program and grids, no re-prefill
    req = urllib.request.Request(API, data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=max(30, remaining - 10)) as r:
        resp = json.load(r)
    msg = resp["choices"][0]["message"]
    _last.reasoning = msg.get("reasoning") or msg.get("reasoning_content") or ""
    _last.usage = resp.get("usage") or {}
    _last.finish = resp["choices"][0].get("finish_reason")
    _last.ids = (resp.get("prompt_token_ids") or []) + (resp["choices"][0].get("token_ids") or []) \
        if resp.get("prompt_token_ids") and resp["choices"][0].get("token_ids") else None
    return msg.get("content") or ""


V14 = bool(os.environ.get("QWEN_V14"))
FORCE_TAIL = ("\n\nI am out of thinking budget. I will now commit to the most likely rule found above and "
              "write the final output grid(s).\n</think>\n\n")
FORCE_TAIL_PROGRAM = ("\n\nI am out of thinking budget. I will now commit to the most likely rule found above, write it "
                      "as the Python function `transform`, then write the final output grid(s).\n</think>\n\n")
FORCE_PREFILL_PROGRAM = "```python\n"
_tail_ids = {}


def _post(url, body, timeout):
    req = urllib.request.Request(url, data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.load(r)


def tokenize(text):
    """Token ids of a plain string (no BOS/specials added), via the vLLM server; cached."""
    if text not in _tail_ids:
        url = API.replace("/v1/chat/completions", "/tokenize")
        _tail_ids[text] = _post(url, {"model": MODEL, "prompt": text, "add_special_tokens": False}, 60)["tokens"]
    return _tail_ids[text]


def force_answer(user_content, reasoning, deadline_ts, max_tokens=4000, ids=None):
    """Budget forcing: the reasoning hit max_tokens; close the think block ourselves and let the model answer.
    v14: the prompt is the truncated request's own token ids + the closing tail, so with --enable-prefix-caching
    the long trace is not prefilled again (v13 re-prefilled ~40-60k tokens and stalled every running request).
    Fallback: raw /v1/completions text in the Qwen chat format (<|im_start|>role ... <|im_end|>)."""
    remaining = deadline_ts - time.time()
    if remaining < 60:
        raise TimeoutError("deadline")
    program = V14 and os.environ.get("QWEN_PROGRAM")
    tail = (FORCE_TAIL_PROGRAM + FORCE_PREFILL_PROGRAM) if program else FORCE_TAIL
    prompt = None
    if V14 and ids:
        try:
            prompt = ids + tokenize(tail)
        except Exception:
            prompt = None
    if prompt is None:
        prompt = (f"<|im_start|>user\n{user_content}<|im_end|>\n<|im_start|>assistant\n<think>\n"
                  f"{reasoning.rstrip()}{tail}")
    body = {"model": MODEL, "prompt": prompt, "max_tokens": max_tokens, "temperature": 0.3}
    resp = _post(API.replace("/chat/completions", "/completions"), body, max(30, remaining - 10))
    text = resp["choices"][0].get("text") or ""
    return (FORCE_PREFILL_PROGRAM + text) if program else text, (resp.get("usage") or {}).get("completion_tokens")


def solve_task(tid, task, out_dir, deadline_ts, rounds=3, max_tokens=32000, chat_fn=chat):
    t0 = time.time()
    messages = [{"role": "user", "content": build_prompt(task)}]
    rec = {"task": tid, "verified": False, "tests": [None] * len(task["test"]), "rounds": 0, "seconds": 0}
    for r in range(rounds):
        try:
            text = chat_fn(messages, deadline_ts, max_tokens=max_tokens)
        except Exception as e:
            rec["error"] = repr(e)[:200]
            break
        rec["rounds"] = r + 1
        code = extract_code(text)
        if not code:
            messages += [{"role": "assistant", "content": text[-4000:]},
                         {"role": "user", "content": "Reply with the complete function in one ```python block."}]
            continue
        outs = run_program(code, [p["input"] for p in task["train"]])
        if all(o == p["output"] for o, p in zip(outs, task["train"])):
            rec["verified"] = True
            rec["tests"] = run_program(code, [t["input"] for t in task["test"]])
            rec["code"] = code
            break
        messages += [{"role": "assistant", "content": f"```python\n{code}\n```"},
                     {"role": "user", "content": FEEDBACK.format(report=diff_report(task, outs))}]
    rec["seconds"] = round(time.time() - t0, 1)
    with open(os.path.join(out_dir, f"{tid}.json"), "w") as f:
        json.dump(rec, f)
    return rec


def task_size(task):
    return sum(len(g) * len(g[0]) for p in task["train"] for g in p.values()) + \
        sum(len(t["input"]) * len(t["input"][0]) for t in task["test"])


def solve_all(tasks, out_dir, deadline_ts, workers=16, chat_fn=chat, solver=None, first=(), **kw):
    solver = solver or solve_task
    os.makedirs(out_dir, exist_ok=True)
    done = {f[:-5] for f in os.listdir(out_dir) if f.endswith(".json")}  # traces are .trace.txt
    # small first: more solved per hour; `first` (e.g. tasks NVARC never reached) ahead of everything
    order = sorted((k for k in tasks if k not in done), key=lambda k: (k not in first, task_size(tasks[k])))
    if solver is solve_direct and os.environ.get("QWEN_REPAIR"):
        return _solve_with_repair(tasks, order, out_dir, deadline_ts, workers, chat_fn, **kw)
    with ThreadPoolExecutor(workers) as ex:
        futs = [ex.submit(lambda k=k: solver(k, tasks[k], out_dir, deadline_ts, chat_fn=chat_fn, **kw)
                          if time.time() < deadline_ts - 60 else None) for k in order]
        for f in futs:
            try:
                f.result()
            except Exception as e:
                print("task error", e, flush=True)


def _needs_repair(rec):
    return bool(rec and rec.get("code") and not rec.get("verified") and "repair" not in rec)


def _solve_with_repair(tasks, order, out_dir, deadline_ts, workers, chat_fn, max_tokens=64000, **_):
    """Workers take round-1 tasks first (coverage), then repairs, best train_ok fraction first. QWEN_REPAIR=eager
    lets near-misses (>= half the training pairs right) jump ahead of untouched tasks."""
    lock = threading.Lock()
    first, fixq, busy = list(order), [], [0]
    eager = os.environ.get("QWEN_REPAIR") == "eager"
    rtok = int(os.environ.get("QWEN_REPAIR_TOKENS", "24000"))

    def push(rec):
        if _needs_repair(rec):
            fixq.append((rec["train_ok"] / len(tasks[rec["task"]]["train"]), -task_size(tasks[rec["task"]]), rec["task"]))
            fixq.sort(reverse=True)

    for f in os.listdir(out_dir):  # resumed run: repairs left over from earlier work
        if f.endswith(".json") and f[:-5] in tasks:
            push(json.load(open(os.path.join(out_dir, f))))

    def worker():
        while time.time() < deadline_ts - 60:
            with lock:
                if fixq and (not first or (eager and fixq[0][0] >= 0.5)):
                    job = ("fix", fixq.pop(0)[2])
                elif first:
                    job = ("r1", first.pop(0)); busy[0] += 1
                elif busy[0] == 0:
                    return
                else:
                    job = None
            if job is None:
                time.sleep(5)
                continue
            kind, k = job
            try:
                if kind == "r1":
                    rec = solve_direct(k, tasks[k], out_dir, deadline_ts, max_tokens=max_tokens, chat_fn=chat_fn)
                    with lock:
                        push(rec)
                else:
                    repair_task(k, tasks[k], out_dir, deadline_ts, max_tokens=rtok, chat_fn=chat_fn)
            except Exception as e:
                print("task error", k, kind, repr(e), flush=True)
            finally:
                if kind == "r1":
                    with lock:
                        busy[0] -= 1

    ts = [threading.Thread(target=worker, daemon=True) for _ in range(workers)]
    for t in ts:
        t.start()
    for t in ts:
        t.join()


DIRECT = """You are solving an ARC-AGI puzzle. Each grid is written as rows of digits (colors 0-9).
Study the training examples, infer the transformation, and apply it to the test input.

{pairs}

{tests}

{style}At the end, give the output grid for each test input, in order, each in
its own ``` block containing only rows of digits (no spaces, no other text)."""
STYLES = {
    "": "Think it through carefully. ",
    "concise": ("Think efficiently: form a hypothesis, check it against every training pair, fix it if needed, "
                "and stop as soon as one rule explains all pairs. Do not re-verify cells you already checked. "),
}
PROGRAM_NOTE = ("Before the final grids, also write the rule as a complete, self-contained Python 3 function "
                "`def transform(grid):` (grid is a list of rows of ints; return a list of rows) in one ```python "
                "block, using only the standard library. It will be run on every training input to check it.\n")
PROGRAM_NOTE_V14 = ("Before the final grids, also write the rule as a complete, self-contained Python 3 function "
                    "`def transform(grid):` (grid is a list of rows of ints; return a list of rows) in one ```python "
                    "block. You may use the standard library and numpy (`import numpy as np`). It will be run on every "
                    "training input and must reproduce every training output exactly.\n")
REPAIR = """I ran your `transform` on the training inputs:
{report}
Find what is wrong with the rule or with the code. Then reply with the complete corrected `def transform(grid):` in one
```python block (standard library and numpy only), followed by the output grid for each test input, in order, each in
its own ``` block containing only rows of digits."""


def _dims(g):
    return f"{len(g)}x{len(g[0])}"


def sane(task, outs):
    """Reject train-verified programs whose test output is clearly degenerate: the unchanged test input when no
    training pair is an identity, or a single colour when every training output has several colours."""
    for t, o in zip(task["test"], outs):
        if o == t["input"] and all(p["input"] != p["output"] for p in task["train"]):
            return False
        if len({v for r in o for v in r}) == 1 and all(len({v for r in p["output"] for v in r}) > 1 for p in task["train"]):
            return False
    return True


def check_program(task, code, deadline_ts):
    """Run code on train + test inputs -> (train_ok, test outputs or None if not verified, train outputs, train errors)."""
    n = len(task["train"])
    outs, errs = run_program_err(code, [p["input"] for p in task["train"]] + [t["input"] for t in task["test"]],
                                 timeout=max(1, min(20, deadline_ts - time.time() - 5)))
    ok = sum(o == p["output"] for o, p in zip(outs, task["train"]))
    tests = outs[n:]
    good = ok == n and all(o is not None for o in tests) and (not V14 or sane(task, tests))
    return ok, (tests if good else None), outs[:n], errs[:n]


def parse_grids(text):
    grids = []
    for b in re.findall(r"```[a-z]*\s*\n(.*?)```", text or "", re.S):
        rows = [r.strip() for r in b.strip().splitlines() if r.strip()]
        if rows and all(re.fullmatch(r"[0-9]+", r) for r in rows) and len({len(r) for r in rows}) == 1 \
                and len(rows) <= 30 and len(rows[0]) <= 30:
            grids.append([[int(c) for c in r] for r in rows])
    return grids


def solve_direct(tid, task, out_dir, deadline_ts, max_tokens=64000, chat_fn=chat, **_):
    t0 = time.time()
    dim = (lambda g: f" ({_dims(g)})") if V14 else (lambda g: "")  # v14: sizes up front, so the model need not count
    pairs = "\n\n".join(f"Example {i+1} input{dim(p['input'])}:\n{grid_text(p['input'])}\n"
                        f"Example {i+1} output{dim(p['output'])}:\n{grid_text(p['output'])}"
                        for i, p in enumerate(task["train"]))
    tests = "\n\n".join(f"Test {i+1} input{dim(t['input'])}:\n{grid_text(t['input'])}" for i, t in enumerate(task["test"]))
    rec = {"task": tid, "tests": [None] * len(task["test"]), "seconds": 0}
    # deadline-aware budget: never start a generation that cannot finish (L4 ~20 tok/s per request at 7-8 concurrent)
    tps = float(os.environ.get("QWEN_TOKS_PER_S", "0"))
    if tps > 0:
        max_tokens = min(max_tokens, int((deadline_ts - time.time() - 90) * tps))
        if max_tokens < int(os.environ.get("QWEN_MIN_TOKENS", "8000")):
            return None  # not enough time left for a useful attempt; leave the task to NVARC
        rec["budget"] = max_tokens
    try:
        user = DIRECT.format(pairs=pairs, tests=tests, style=STYLES[os.environ.get("QWEN_STYLE", "")])
        if os.environ.get("QWEN_PROGRAM"):
            user = user.replace("At the end, give", (PROGRAM_NOTE_V14 if V14 else PROGRAM_NOTE) + "At the end, give")
        _last.ids = None
        text = chat_fn([{"role": "user", "content": user}], deadline_ts, max_tokens=max_tokens)
        rec["completion_tokens"] = (getattr(_last, "usage", {}) or {}).get("completion_tokens")
        rec["finish"] = getattr(_last, "finish", None)
        if rec["finish"] == "length" and os.environ.get("QWEN_FORCE") and not parse_grids(text):
            try:  # salvage a truncated reasoning trace instead of throwing the whole generation away
                tf = time.time()
                text, rec["forced_tokens"] = force_answer(user, getattr(_last, "reasoning", "") or text, deadline_ts,
                                                          max_tokens=6000 if V14 else 4000, ids=getattr(_last, "ids", None))
                rec["forced"] = True
                rec["force_s"] = round(time.time() - tf, 1)
                rec["force_ids"] = bool(getattr(_last, "ids", None))
            except Exception as e:
                rec["force_error"] = repr(e)[:200]
        if os.environ.get("QWEN_SAVE_TRACES"):
            with open(os.path.join(out_dir, f"{tid}.trace.txt"), "w") as f:
                f.write(getattr(_last, "reasoning", "") + "\n<<<ANSWER>>>\n" + text)
        g = parse_grids(text)
        n = len(task["test"])
        rec["tests"] = (g[-n:] + [None] * n)[:n] if g else rec["tests"]
        rec["chars"] = len(text)
        code = extract_code(text) if os.environ.get("QWEN_PROGRAM") else None
        if code:  # v13: a program that reproduces every train pair outranks the written grids
            rec["train_ok"], vt, _, errs = check_program(task, code, deadline_ts)
            if vt is not None:
                rec["verified"] = True
                rec["written"] = rec["tests"]
                rec["tests"] = vt
            elif V14:  # kept for the repair round
                rec["code"], rec["content"] = code, text
                rec["suspect"] = rec["train_ok"] == len(task["train"])
    except Exception as e:
        rec["error"] = repr(e)[:200]
    rec["seconds"] = round(time.time() - t0, 1)
    save_rec(out_dir, tid, rec)
    return rec


def repair_task(tid, task, out_dir, deadline_ts, max_tokens=24000, chat_fn=chat, **_):
    """Round 2 for an unverified program: show it what its transform() produced on each training pair and ask for a
    fixed version. A repaired program that verifies replaces the round-1 grids (round-1 grids are kept in 'written')."""
    path = os.path.join(out_dir, f"{tid}.json")
    rec = json.load(open(path))
    t0 = time.time()
    tps = float(os.environ.get("QWEN_TOKS_PER_S", "0"))
    if tps > 0:
        max_tokens = min(max_tokens, int((deadline_ts - time.time() - 90) * tps))
        if max_tokens < int(os.environ.get("QWEN_MIN_TOKENS", "8000")):
            return None
    rep = {"budget": max_tokens}
    try:
        n = len(task["train"])
        _, _, outs, errs = check_program(task, rec["code"], deadline_ts)
        dim = lambda g: f" ({_dims(g)})"
        pairs = "\n\n".join(f"Example {i+1} input{dim(p['input'])}:\n{grid_text(p['input'])}\n"
                            f"Example {i+1} output{dim(p['output'])}:\n{grid_text(p['output'])}"
                            for i, p in enumerate(task["train"]))
        tests = "\n\n".join(f"Test {i+1} input{dim(t['input'])}:\n{grid_text(t['input'])}" for i, t in enumerate(task["test"]))
        user = DIRECT.format(pairs=pairs, tests=tests, style=STYLES[os.environ.get("QWEN_STYLE", "")])
        user = user.replace("At the end, give", PROGRAM_NOTE_V14 + "At the end, give")
        fb = REPAIR.format(report=diff_report(task, outs, errs))
        msgs = [{"role": "user", "content": user}, {"role": "assistant", "content": rec["content"]},
                {"role": "user", "content": fb}]
        _last.ids = None
        text = chat_fn(msgs, deadline_ts, max_tokens=max_tokens)
        rep["completion_tokens"] = (getattr(_last, "usage", {}) or {}).get("completion_tokens")
        rep["finish"] = getattr(_last, "finish", None)
        if rep["finish"] == "length" and not extract_code(text) and getattr(_last, "ids", None):  # needs exact ids
            try:
                text, rep["forced_tokens"] = force_answer(fb, getattr(_last, "reasoning", "") or text, deadline_ts,
                                                          max_tokens=6000, ids=getattr(_last, "ids", None))
                rep["forced"] = True
            except Exception as e:
                rep["force_error"] = repr(e)[:200]
        code = extract_code(text)
        if code:
            rep["train_ok"], vt, _, _ = check_program(task, code, deadline_ts)
            if vt is not None:
                rep["verified"] = True
                rec["verified"] = True
                rec["repaired"] = True
                rec["written"] = rec["tests"]
                rec["tests"] = vt
        g = parse_grids(text)
        k = len(task["test"])
        rep["tests"] = (g[-k:] + [None] * k)[:k] if g else None  # repair-round written grids, analysis only
    except Exception as e:
        rep["error"] = repr(e)[:200]
    rep["seconds"] = round(time.time() - t0, 1)
    rec["repair"] = rep
    save_rec(out_dir, tid, rec)
    return rec


def save_rec(out_dir, tid, rec):  # write-then-rename: readers never see a half-written file
    tmp = os.path.join(out_dir, f".{tid}.tmp")
    with open(tmp, "w") as f:
        json.dump(rec, f)
    os.replace(tmp, os.path.join(out_dir, f"{tid}.json"))


def merge_with_nvarc(submission, out_dir):
    """Verified program output -> attempt_1 (NVARC best -> attempt_2). Returns #tests overridden."""
    n = 0
    for f in os.listdir(out_dir):
        if not f.endswith(".json"):
            continue
        rec = json.load(open(os.path.join(out_dir, f)))
        if not rec.get("verified") or rec["task"] not in submission:
            continue
        for i, g in enumerate(rec["tests"]):
            if g is None or i >= len(submission[rec["task"]]):
                continue
            slot = submission[rec["task"]][i]
            if slot["attempt_1"] != g:
                slot["attempt_2"] = slot["attempt_1"]
                slot["attempt_1"] = g
                n += 1
    return n


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--tasks", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--deadline-ts", type=float, required=True)
    ap.add_argument("--workers", type=int, default=16)
    ap.add_argument("--rounds", type=int, default=3)
    ap.add_argument("--max-tokens", type=int, default=32000)
    a = ap.parse_args()
    solve_all(json.load(open(a.tasks)), a.out, a.deadline_ts, workers=a.workers, rounds=a.rounds, max_tokens=a.max_tokens)


Writing /kaggle/working/qwen_solver.py


In [4]:
!pip uninstall -y tensorflow

Found existing installation: tensorflow 2.18.0
Uninstalling tensorflow-2.18.0:
  Successfully uninstalled tensorflow-2.18.0


In [5]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

Writing arc_loader.py


In [6]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

Writing arc_decoder.py


In [7]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

import glob as _glob
_cands = sorted(os.path.dirname(c) for c in _glob.glob("/kaggle/input/**/config.json", recursive=True) if "qwen3_4b_grids15_sft139" in c)
assert _cands, "qwen3_4b_grids15_sft139 not found under /kaggle/input"
MODEL_PATH = _cands[0]
print("MODEL_PATH", MODEL_PATH)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for t in ARC_TOKENS:
            score = nll[i, t].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    batch_logits = outputs.logits.float().cpu().log_softmax(-1)
    result = []
    for logits, query_tokens, answer_tokens in zip(batch_logits, batch_query_tokens, batch_answer_tokens):
        query_length = len(query_tokens)
        answer_logits = logits[query_length-1:query_length-1+len(answer_tokens)]
        answer_score = answer_logits[torch.arange(len(answer_tokens)), answer_tokens].sum()
        result.append(-answer_score.item())
    return result


def worker(rank, queue, end_time):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    if rerun_mode:
        test_path = "/kaggle/working/arcdata/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/working/arcdata/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break
        
        start_time = time.time()
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=8, shfl_keys=True, seed=1)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=1, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        _per = defaultdict(lambda: defaultdict(list))
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            _per[test_id][len(tokenizer.encode(eval_ds.get(subkey, formatter)["input"]))].append(subkey)
        _chunks = {}
        for test_id, groups in _per.items():
            _cls = []
            for _len, items in sorted(groups.items(), key=lambda kv: (-len(kv[1]), kv[0])):
                _rows = int(os.environ.get("ARC_MAX_ROWS", "8")) if _len <= int(os.environ.get("ARC_ROWS8_MAX_CTX", "3600")) else 4
                _cls.append([items[i:i + _rows] for i in range(0, len(items), _rows)])
            _chunks[test_id] = [c[i] for i in range(max(len(c) for c in _cls)) for c in _cls if i < len(c)]
        batches = [_chunks[t][i] for i in range(max(len(c) for c in _chunks.values())) for t in sorted(_chunks) if i < len(_chunks[t])]
        print(f"[Rank {rank}] DFS8 {key}: {len(batches)} batches rows {[len(b) for b in batches]}", flush=True)

        with torch.inference_mode():
                
            known_scores = {}

            for subkeys in batches:

                spend_time = time.time() - start_time
                if spend_time > 1200 or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                _split = False
                if len(tokens) > 4:
                    _need = len(tokens) * (max(len(t) for t in tokens) + max_new_tokens) * 147456 * 5 // 2 + (2 << 30)
                    _spare = torch.cuda.mem_get_info()[0] + torch.cuda.memory_reserved() - torch.cuda.memory_allocated()
                    _split = _spare < _need
                    if _split:
                        print(f"[Rank {rank}] DFS8 split {len(tokens)} rows (spare {_spare >> 20}MB need {_need >> 20}MB)", flush=True)
                if not _split:
                    try:
                        dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)
                    except (torch.OutOfMemoryError, RuntimeError) as _e:
                        if len(tokens) <= 4 or not ('out of memory' in str(_e) or 'CUBLAS' in str(_e)):
                            raise
                        print(f"[Rank {rank}] DFS8 OOM on {len(tokens)} rows, retry as 4+rest: {str(_e)[:80]}", flush=True)
                        gc.collect(); torch.cuda.empty_cache()
                        _split = True
                if _split:
                    dfs_result = list(inference_turbo_dfs(model, tokens[:4], max_new_tokens, max_score, end_time))
                    gc.collect(); torch.cuda.empty_cache()
                    dfs_result += [(4 + _i, _b) for _i, _b in inference_turbo_dfs(model, tokens[4:], max_new_tokens, max_score, end_time)]

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")

Writing arc_solver.py


In [8]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time):
    
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Fix Unsloth patching issue
    if rank > 0:
        while not os.path.exists(f"/kaggle/worker{rank-1}"):
            time.sleep(5)
    
    from arc_solver import worker

    with open(f"/kaggle/worker{rank}", "w") as f:
        f.write("Ok")
    
    print(f"[Rank {rank}] start!")
    
    worker(rank, queue, end_time)
    
    print(f"[Rank {rank}] done!")


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if rerun_mode:
        test_path = "/kaggle/working/arcdata/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/working/arcdata/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    legc_skip = set()
    legc_path = "/kaggle/working/induction_results.json"
    if os.path.exists(legc_path):
        with open(legc_path) as f:
            legc_skip = {k for k, r in json.load(f).items()
                         if isinstance(r, dict) and r.get("verified")}
        print(f"[LegC] skipping {len(legc_skip)} verified task(s) in the base queue")

    queue = mp.Manager().Queue()

    for key in sorted(data.keys()):
        if not rerun_mode:
            if key not in ["0934a4d8", "36a08778", "981571dc", "aa4ec2a5"]:
                continue
        if key in legc_skip:
            continue
        queue.put(key)
    for _ in range(4):
        queue.put(None)
    
    mp.spawn(local_worker, args=(queue, args.end_time), nprocs=4)

Writing starter.py


In [9]:
%%writefile arc_induction_v2.py
#!/usr/bin/env python3
# =============================================================================
#  ARC-AGI-2  Leg C : Verified-Induction Pre-Pass  (v2, base-notebook-aligned)
#  -------------------------------------------------------------
#  Role      : runs BEFORE the base 4B TTT pipeline (failed-in-aimo line).
#              Samples DSL programs with a 7B coder (one per L4, 4 workers),
#              verifies them against ALL train pairs in an isolated,
#              torch-free sandbox, and emits train-verified test outputs.
#              A separate merge cell then lets verified outputs pre-empt
#              attempt_1 while attempt_2 keeps the base (transduction)
#              candidate -> monotonic: final score can only add coverage.
#  Env facts : Kaggle AGI-2 = Python 3.11, L4 22GB x4, offline.
#              -> NO vLLM / cp312 wheelhouse dependency (27B v4 incident).
#              -> 7B bf16 fits a single L4; 4 independent workers.
#              -> heavy imports live INSIDE worker fns (spawn re-import cheap).
#  Fixes vs the "hacker plan":
#     1. exec single-namespace (two-dict exec made the DSL unreachable)
#     2. `import re` present; no undefined names in the submit loop
#     3. DSL is ENFORCED via AST whitelist, not politely requested
#     4. 35-primitive library instead of 3; per-program SIGALRM timeout
#     5. batched sampling (K per call) + 1 repair round, not 3 blind shots
#     6. official 2026 submission format {tid: [{attempt_1, attempt_2}, ..]}
#  Usage (Kaggle):
#     !python arc_induction_v1.py --budget-h 1.5
#     !python arc_induction_v1.py --selftest        # sandbox unit tests, no GPU
#     !python arc_induction_v1.py --smoke 4         # 4 easiest tasks, verbose
# =============================================================================

import os
import sys
import re
import json
import time
import math
import random
import hashlib
import argparse
import subprocess
import multiprocessing as mp

# ----------------------------------------------------------------------------
# Paths / defaults
# ----------------------------------------------------------------------------
DATA_DIR_CANDIDATES = [
    "/kaggle/working/arcdata",  # path used by the LB-proven base notebook
    "/kaggle/input/arc-prize-2026-arc-agi-2",               # standard mount fallback
]
WORK_DIR      = os.environ.get("ARC_WORK_DIR", "/kaggle/working")
RUNTIME_PATH  = os.path.join(WORK_DIR, "_dsl_runtime.py")
RESULTS_PATH  = os.path.join(WORK_DIR, "induction_results.json")
AUDIT_PATH    = os.path.join(WORK_DIR, "induction_audit.json")
SUB_PATH      = os.path.join(WORK_DIR, "submission_induction_only.json")
PARTIAL_TPL   = os.path.join(WORK_DIR, "_legc_partial_{rank}.jsonl")

MODEL_PATH_CANDIDATES = [
    "/kaggle/input/models/qwen-lm/qwen2.5-coder/transformers/7b-instruct/1",  # user's attach path
    "/kaggle/input/qwen-lm/qwen2.5-coder/transformers/7b-instruct/1",
    "/kaggle/input/qwen2.5-coder-7b-instruct/transformers/default/1",
    "/kaggle/input/qwen2.5-coder/transformers/7b-instruct/1",
    "/kaggle/input/qwen2.5-coder-7b-instruct",
]

DEFAULTS = dict(
    budget_h        = 1.5,    # wall budget of the whole induction phase
    per_task_cap    = 200.0,  # seconds per task (both rounds included)
    k1              = 8,      # round-1 samples per task
    k2              = 6,      # repair-round samples
    temp1           = 0.8,
    temp2           = 0.7,
    max_new_tokens  = 640,
    max_prompt_chars= 26000,
    prog_timeout    = 4.0,    # per-candidate CPU seconds in the sandbox
    seed            = 1234,
    print_grids     = 8,      # dump grids for first N verified tasks
)

# ----------------------------------------------------------------------------
# The sandbox runtime (torch-free).  Written to _dsl_runtime.py and executed
# as a subprocess:  python _dsl_runtime.py in.json out.json
# Candidate code is exec'd into ONE namespace that already CONTAINS the DSL
# functions as values -> transform.__globals__ sees them.  (This is the
# structural fix for the two-dict exec bug.)
# ----------------------------------------------------------------------------
RUNTIME_CODE = r'''
import sys, json, ast, signal, builtins
import numpy as np
from scipy.ndimage import label as _cc_label

MAXD = 30
_ST4 = np.array([[0,1,0],[1,1,1],[0,1,0]])
_ST8 = np.ones((3,3), dtype=int)

def _np(g): return np.asarray(g, dtype=int)
def _ls(a): return [[int(x) for x in row] for row in np.asarray(a)]

# ---------------- grid basics ----------------
def grid_shape(g):
    a = _np(g); return (int(a.shape[0]), int(a.shape[1]))
def new_grid(h, w, fill=0):
    return [[int(fill)] * int(w) for _ in range(int(h))]
def rot90(g):  return _ls(np.rot90(_np(g), 3))     # clockwise
def rot180(g): return _ls(np.rot90(_np(g), 2))
def rot270(g): return _ls(np.rot90(_np(g), 1))     # counter-clockwise
def flip_h(g): return _ls(np.fliplr(_np(g)))       # mirror left-right
def flip_v(g): return _ls(np.flipud(_np(g)))       # mirror up-down
def transpose(g): return _ls(_np(g).T)
def crop(g, r, c, h, w):
    return _ls(_np(g)[int(r):int(r)+int(h), int(c):int(c)+int(w)])

# ---------------- colors ----------------
def bg_color(g):
    v, n = np.unique(_np(g), return_counts=True); return int(v[np.argmax(n)])
def palette(g): return [int(x) for x in np.unique(_np(g))]
def color_counts(g):
    v, n = np.unique(_np(g), return_counts=True)
    return {int(a): int(b) for a, b in zip(v, n)}
def replace_color(g, a, b):
    arr = _np(g).copy(); arr[arr == int(a)] = int(b); return _ls(arr)
def swap_colors(g, a, b):
    arr = _np(g).copy(); ma = arr == int(a); mb = arr == int(b)
    arr[ma] = int(b); arr[mb] = int(a); return _ls(arr)
def map_colors(g, mapping):
    arr = _np(g); out = arr.copy()
    for k, v in mapping.items(): out[arr == int(k)] = int(v)
    return _ls(out)
def find_color_cells(g, color):
    ys, xs = np.where(_np(g) == int(color))
    return [(int(r), int(c)) for r, c in zip(ys, xs)]
def crop_to_content(g, bg=None):
    a = _np(g); b = bg_color(g) if bg is None else int(bg)
    ys, xs = np.where(a != b)
    if len(ys) == 0: return _ls(a)
    return _ls(a[ys.min():ys.max()+1, xs.min():xs.max()+1])

# ---------------- assembly ----------------
def paste(dst, src, r, c, transparent=None):
    D = _np(dst).copy(); S = _np(src)
    r0, c0 = int(r), int(c); h, w = S.shape
    r1, c1 = max(r0, 0), max(c0, 0)
    r2, c2 = min(r0 + h, D.shape[0]), min(c0 + w, D.shape[1])
    if r2 <= r1 or c2 <= c1: return _ls(D)
    sub = S[r1-r0:r2-r0, c1-c0:c2-c0]
    if transparent is None:
        D[r1:r2, c1:c2] = sub
    else:
        m = sub != int(transparent)
        D[r1:r2, c1:c2][m] = sub[m]
    return _ls(D)
def concat_h(a, b): return _ls(np.concatenate([_np(a), _np(b)], axis=1))
def concat_v(a, b): return _ls(np.concatenate([_np(a), _np(b)], axis=0))
def tile(g, ny, nx): return _ls(np.tile(_np(g), (int(ny), int(nx))))
def upscale(g, fy, fx):
    return _ls(np.kron(_np(g), np.ones((int(fy), int(fx)), dtype=int)))
def downscale(g, fy, fx):
    a = _np(g); fy, fx = int(fy), int(fx)
    h, w = a.shape[0] // fy, a.shape[1] // fx
    out = np.zeros((h, w), dtype=int)
    for i in range(h):
        for j in range(w):
            blk = a[i*fy:(i+1)*fy, j*fx:(j+1)*fx].ravel()
            v, n = np.unique(blk, return_counts=True)
            out[i, j] = v[np.argmax(n)]
    return _ls(out)
def cellwise(a, b, f):
    A, B = _np(a), _np(b)
    if A.shape != B.shape: raise ValueError("cellwise: shape mismatch")
    return [[int(f(int(A[i, j]), int(B[i, j]))) for j in range(A.shape[1])]
            for i in range(A.shape[0])]
def grid_eq(a, b):
    A, B = _np(a), _np(b)
    return A.shape == B.shape and bool((A == B).all())
def split_rows(g, n):
    a = _np(g); h = a.shape[0] // int(n)
    return [_ls(a[i*h:(i+1)*h, :]) for i in range(int(n))]
def split_cols(g, n):
    a = _np(g); w = a.shape[1] // int(n)
    return [_ls(a[:, i*w:(i+1)*w]) for i in range(int(n))]
def split_by_separator(g):
    # splits on full uniform rows/cols of the dominant separator color;
    # returns row-major list of sub-grids (or [g] when no separator found)
    a = _np(g); H, W = a.shape
    frows = [i for i in range(H) if len(set(a[i, :].tolist())) == 1]
    fcols = [j for j in range(W) if len(set(a[:, j].tolist())) == 1]
    colors = [int(a[i, 0]) for i in frows] + [int(a[0, j]) for j in fcols]
    if not colors: return [_ls(a)]
    v, n = np.unique(np.array(colors), return_counts=True)
    sep = int(v[np.argmax(n)])
    rs = set(i for i in frows if a[i, 0] == sep)
    cs = set(j for j in fcols if a[0, j] == sep)
    def runs(banned, N):
        out, s = [], None
        for i in range(N):
            if i in banned:
                if s is not None: out.append((s, i)); s = None
            else:
                if s is None: s = i
        if s is not None: out.append((s, N))
        return out
    parts = []
    for (r0, r1) in runs(rs, H):
        for (c0, c1) in runs(cs, W):
            parts.append(_ls(a[r0:r1, c0:c1]))
    return parts if len(parts) > 1 else [_ls(a)]

# ---------------- objects ----------------
def _mk_obj(cells):
    rs  = [r for r, _, _ in cells]; csx = [c for _, c, _ in cells]
    col = [v for _, _, v in cells]
    v, n = np.unique(np.array(col), return_counts=True)
    return {'cells': cells,
            'r': min(rs), 'c': min(csx),
            'h': max(rs) - min(rs) + 1, 'w': max(csx) - min(csx) + 1,
            'size': len(cells),
            'color': int(v[np.argmax(n)]),
            'colors': [int(x) for x in v]}
def get_objects(g, bg=None, diagonal=False, multicolor=True):
    a = _np(g); b = bg_color(g) if bg is None else int(bg)
    st = _ST8 if diagonal else _ST4
    objs = []
    if multicolor:
        lab, n = _cc_label(a != b, structure=st)
        for i in range(1, n + 1):
            ys, xs = np.where(lab == i)
            objs.append(_mk_obj([(int(r), int(c), int(a[r, c]))
                                 for r, c in zip(ys, xs)]))
    else:
        for col in palette(g):
            if col == b: continue
            lab, n = _cc_label(a == col, structure=st)
            for i in range(1, n + 1):
                ys, xs = np.where(lab == i)
                objs.append(_mk_obj([(int(r), int(c), col)
                                     for r, c in zip(ys, xs)]))
    return objs
def obj_size(o):  return o['size']
def obj_color(o): return o['color']
def obj_grid(o, bg=0):
    g = np.full((o['h'], o['w']), int(bg), dtype=int)
    for r, c, v in o['cells']: g[r - o['r'], c - o['c']] = v
    return _ls(g)
def subgrid_at(g, o):
    return crop(g, o['r'], o['c'], o['h'], o['w'])
def largest_object(objs):
    return max(objs, key=lambda o: (o['size'], -o['r'], -o['c']))
def smallest_object(objs):
    return min(objs, key=lambda o: (o['size'], o['r'], o['c']))
def paint_objects(g, objs, color):
    a = _np(g).copy()
    for o in objs:
        for r, c, _ in o['cells']: a[r, c] = int(color)
    return _ls(a)
def erase_objects(g, objs, bg=0):
    a = _np(g).copy()
    for o in objs:
        for r, c, _ in o['cells']: a[r, c] = int(bg)
    return _ls(a)
def move_object(g, o, dr, dc, bg=0):
    a = _np(erase_objects(g, [o], bg)); H, W = a.shape
    for r, c, v in o['cells']:
        nr, nc = r + int(dr), c + int(dc)
        if 0 <= nr < H and 0 <= nc < W: a[nr, nc] = v
    return _ls(a)
def fold_repaint(g, objs, color_fn):
    a = _np(g).copy()
    for o in objs:
        col = color_fn(o)
        if col is None: continue
        for r, c, _ in o['cells']: a[r, c] = int(col)
    return _ls(a)
def count_holes(o):
    m = np.zeros((o['h'], o['w']), dtype=int)
    for r, c, _ in o['cells']: m[r - o['r'], c - o['c']] = 1
    lab, n = _cc_label(m == 0, structure=_ST4)
    holes = 0
    for i in range(1, n + 1):
        ys, xs = np.where(lab == i)
        if ys.min() > 0 and xs.min() > 0 and \
           ys.max() < o['h'] - 1 and xs.max() < o['w'] - 1:
            holes += 1
    return holes

# ---------------- physics-ish ----------------
def flood_fill(g, r, c, color):
    a = _np(g).copy(); H, W = a.shape
    r0, c0 = int(r), int(c); tgt = int(a[r0, c0]); col = int(color)
    if tgt == col: return _ls(a)
    stack = [(r0, c0)]
    while stack:
        y, x = stack.pop()
        if 0 <= y < H and 0 <= x < W and a[y, x] == tgt:
            a[y, x] = col
            stack.extend([(y+1, x), (y-1, x), (y, x+1), (y, x-1)])
    return _ls(a)
def gravity(g, direction, bg=None):
    a = _np(g); b = bg_color(g) if bg is None else int(bg)
    H, W = a.shape
    out = np.full_like(a, b)
    if direction in ('down', 'up'):
        for j in range(W):
            col = [v for v in a[:, j].tolist() if v != b]
            if direction == 'down': out[H - len(col):, j] = col
            else:                   out[:len(col), j] = col
    elif direction in ('left', 'right'):
        for i in range(H):
            row = [v for v in a[i, :].tolist() if v != b]
            if direction == 'left': out[i, :len(row)] = row
            else:                   out[i, W - len(row):] = row
    else:
        raise ValueError("gravity: direction must be up/down/left/right")
    return _ls(out)
def is_symmetric_h(g): return grid_eq(g, flip_h(g))
def is_symmetric_v(g): return grid_eq(g, flip_v(g))

EXPORTS = {k: v for k, v in list(globals().items())
           if not k.startswith('_') and callable(v)
           and k not in ('main',)}

# ---------------- enforcement ----------------
_ALLOWED_BUILTINS = ['abs','all','any','bool','dict','divmod','enumerate',
                     'filter','float','frozenset','int','isinstance','len',
                     'list','map','max','min','pow','print','range','repr',
                     'reversed','round','set','sorted','str','sum','tuple','zip']
SAFE_BUILTINS = {k: getattr(builtins, k) for k in _ALLOWED_BUILTINS}

_BANNED_NODES = (ast.While, ast.Import, ast.ImportFrom, ast.Global,
                 ast.Nonlocal, ast.ClassDef, ast.AsyncFunctionDef,
                 ast.AsyncFor, ast.AsyncWith, ast.Await)
_BANNED_NAMES = {'exec','eval','open','__import__','compile','globals',
                 'locals','vars','getattr','setattr','delattr','input',
                 'breakpoint','exit','quit','help','super','type','object',
                 'memoryview','bytearray','bytes'}

def validate_source(src):
    try:
        tree = ast.parse(src)
    except SyntaxError as e:
        return 'syntax error: %s' % e
    has_transform = False
    for node in ast.walk(tree):
        if isinstance(node, _BANNED_NODES):
            return 'banned construct: %s' % type(node).__name__
        if isinstance(node, ast.Name) and node.id in _BANNED_NAMES:
            return 'banned name: %s' % node.id
        if isinstance(node, ast.Attribute) and node.attr.startswith('__'):
            return 'banned dunder attribute: %s' % node.attr
        if isinstance(node, ast.FunctionDef) and node.name == 'transform':
            has_transform = True
    if not has_transform:
        return 'missing def transform(grid)'
    return None

class _Timeout(Exception): pass
def _on_alarm(signum, frame): raise _Timeout()
signal.signal(signal.SIGALRM, _on_alarm)

def _norm_out(g):
    a = np.asarray(g)
    if a.dtype == object or a.ndim != 2 or a.size == 0:
        raise ValueError('output must be a non-empty rectangular 2D grid')
    if a.shape[0] > MAXD or a.shape[1] > MAXD:
        raise ValueError('output larger than %dx%d' % (MAXD, MAXD))
    a = a.astype(int)
    if a.min() < 0 or a.max() > 9:
        raise ValueError('colors must be integers 0..9')
    return a.tolist()

def _copy_grid(g): return [row[:] for row in g]

def run_one(src, train, test, tmo):
    err = validate_source(src)
    if err:
        return {'status': 'rejected', 'error': err, 'pairs_passed': 0}
    env = dict(EXPORTS)
    env['np'] = np                      # numpy allowed read-only by convention
    env['__builtins__'] = SAFE_BUILTINS
    try:
        signal.setitimer(signal.ITIMER_REAL, float(tmo))
        exec(compile(src, '<candidate>', 'exec'), env)   # SINGLE namespace
        tf = env.get('transform')
        if not callable(tf):
            return {'status': 'rejected', 'error': 'transform not callable',
                    'pairs_passed': 0}
        pairs_passed, fail = 0, None
        for i, p in enumerate(train):
            try:
                got = _norm_out(tf(_copy_grid(p['input'])))
            except _Timeout:
                raise
            except Exception as e:
                if fail is None:
                    fail = {'idx': i, 'err': repr(e)[:200]}
                continue
            if got == p['output']:
                pairs_passed += 1
            elif fail is None:
                fail = {'idx': i, 'got': got}
        if pairs_passed == len(train):
            touts = [_norm_out(tf(_copy_grid(t))) for t in test]
            return {'status': 'verified', 'pairs_passed': pairs_passed,
                    'n_pairs': len(train), 'test_outputs': touts}
        out = {'status': 'partial', 'pairs_passed': pairs_passed,
               'n_pairs': len(train)}
        if fail: out.update(fail)
        return out
    except _Timeout:
        return {'status': 'timeout', 'pairs_passed': 0,
                'error': 'exceeded %.1fs' % tmo}
    except Exception as e:
        return {'status': 'error', 'pairs_passed': 0,
                'error': repr(e)[:300]}
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)

def main():
    sys.setrecursionlimit(300)
    inp, outp = sys.argv[1], sys.argv[2]
    with open(inp) as f:
        req = json.load(f)
    results = [run_one(src, req['train'], req['test'], req.get('timeout', 4.0))
               for src in req['programs']]
    with open(outp, 'w') as f:
        json.dump({'results': results}, f)

if __name__ == '__main__':
    main()
'''

# ----------------------------------------------------------------------------
# Prompting
# ----------------------------------------------------------------------------
SYSTEM_MSG = (
    "You are an expert ARC puzzle solver. You write short Python solutions "
    "using ONLY the provided DSL helpers plus basic pure Python "
    "(for-loops, comprehensions, if/else, lambda). Follow the rules exactly."
)

CHEAT_SHEET = """AVAILABLE DSL (grids are list[list[int]], colors 0-9):
grid_shape(g)->(h,w) | new_grid(h,w,fill=0) | crop(g,r,c,h,w)
rot90(g) rot180(g) rot270(g) | flip_h(g) flip_v(g) | transpose(g)
bg_color(g) | palette(g) | color_counts(g)->dict | find_color_cells(g,color)
replace_color(g,a,b) | swap_colors(g,a,b) | map_colors(g,{a:b,...})
crop_to_content(g,bg=None)
paste(dst,src,r,c,transparent=None) | concat_h(a,b) concat_v(a,b)
tile(g,ny,nx) | upscale(g,fy,fx) | downscale(g,fy,fx)
cellwise(a,b,f) | grid_eq(a,b) | split_rows(g,n) split_cols(g,n)
split_by_separator(g)->list of sub-grids (row-major)
get_objects(g,bg=None,diagonal=False,multicolor=True)->list of obj
  obj = {'cells':[(r,c,color)..],'r','c','h','w','size','color','colors'}
obj_size(o) obj_color(o) | obj_grid(o,bg=0) | subgrid_at(g,o)
largest_object(objs) smallest_object(objs)
paint_objects(g,objs,color) | erase_objects(g,objs,bg=0)
move_object(g,o,dr,dc,bg=0) | fold_repaint(g,objs,color_fn)
count_holes(o) | flood_fill(g,r,c,color) | gravity(g,'down'|'up'|'left'|'right')
is_symmetric_h(g) is_symmetric_v(g)

RULES:
- Define exactly one function: def transform(grid) -> grid.
- NO imports. NO while loops. NO classes. Use only helpers above + pure python.
- Return a new grid (list of lists of ints 0-9), never mutate weird state.
"""

FEWSHOT = """Two tiny examples of the expected style:

Example task: recolor every object to the color of the largest object.
```python
def transform(grid):
    objs = get_objects(grid)
    big = largest_object(objs)
    return fold_repaint(grid, objs, lambda o: obj_color(big))
```

Example task: output = input rotated 90 degrees clockwise, colors 1<->3 swapped.
```python
def transform(grid):
    return swap_colors(rot90(grid), 1, 3)
```
"""

def render_grid(g):
    return "\n".join("".join(str(int(v)) for v in row) for row in g)

def shape_hint(task):
    ins  = [(len(p['input']),  len(p['input'][0]))  for p in task['train']]
    outs = [(len(p['output']), len(p['output'][0])) for p in task['train']]
    if all(a == b for a, b in zip(ins, outs)):
        return "Observed shape rule: every output has the SAME shape as its input."
    if len(set(outs)) == 1:
        h, w = outs[0]
        return f"Observed shape rule: every output is exactly {h}x{w}."
    ratios = set()
    ok = True
    for (ih, iw), (oh, ow) in zip(ins, outs):
        if oh % ih == 0 and ow % iw == 0:
            ratios.add((oh // ih, ow // iw))
        else:
            ok = False
    if ok and len(ratios) == 1:
        fy, fx = next(iter(ratios))
        return f"Observed shape rule: output = input scaled by ({fy}x, {fx}x)."
    return "Observed shape rule: varies -- infer it from the pairs. " + \
           f"in->out shapes: {list(zip(ins, outs))}"

def build_prompt(task, repair=None):
    lines = [CHEAT_SHEET, FEWSHOT, "--- TASK ---"]
    for i, p in enumerate(task['train'], 1):
        ih, iw = len(p['input']), len(p['input'][0])
        oh, ow = len(p['output']), len(p['output'][0])
        lines.append(f"Train {i} Input ({ih}x{iw}):\n{render_grid(p['input'])}")
        lines.append(f"Train {i} Output ({oh}x{ow}):\n{render_grid(p['output'])}")
    for j, t in enumerate(task['test'], 1):
        th, tw = len(t['input']), len(t['input'][0])
        lines.append(f"Test {j} Input ({th}x{tw}):\n{render_grid(t['input'])}")
    lines.append(shape_hint(task))
    if repair:
        lines.append(
            "### PREVIOUS ATTEMPT (best so far) ###\n"
            f"```python\n{repair['code']}\n```\n"
            f"It passed {repair['pairs_passed']}/{repair['n_pairs']} train pairs "
            f"but FAILED on Train {repair['fail_idx'] + 1}.\n"
            f"{repair['diff']}\n"
            "Fix the LOGIC. Keep the DSL rules. Output the corrected full solution."
        )
    lines.append("Think briefly, then answer with ONE ```python code block "
                 "containing def transform(grid).")
    return "\n\n".join(lines)

def repair_diff(task, part):
    i = part.get('idx', part.get('fail_idx', 0))
    exp = task['train'][i]['output']
    if 'got' in part and part['got'] is not None:
        got = part['got']
        gh, gw = len(got), len(got[0])
        eh, ew = len(exp), len(exp[0])
        if (gh, gw) != (eh, ew):
            return f"Your output shape was {gh}x{gw}, expected {eh}x{ew}."
        if gh <= 15 and gw <= 15:
            return ("Expected:\n" + render_grid(exp) +
                    "\nYour output:\n" + render_grid(got))
        wrong = sum(1 for r in range(eh) for c in range(ew)
                    if got[r][c] != exp[r][c])
        return f"Shape correct ({eh}x{ew}) but {wrong} cells are wrong."
    if 'err' in part:
        return f"It raised: {part['err']}"
    return "Output did not match."

def extract_code(text):
    m = re.findall(r"```python(.*?)```", text, re.DOTALL)
    if m: return m[-1].strip()
    m = re.findall(r"```(.*?)```", text, re.DOTALL)
    if m: return m[-1].strip()
    idx = text.find("def transform")
    return text[idx:].strip() if idx >= 0 else text.strip()

def code_key(src):
    return hashlib.md5(re.sub(r"\s+", "", src).encode()).hexdigest()

# ----------------------------------------------------------------------------
# Sandbox client (used by workers AND selftest; torch-free)
# ----------------------------------------------------------------------------
def run_verifier(programs, train, test, prog_timeout, tag):
    inp  = os.path.join(WORK_DIR, f"_vin_{tag}.json")
    outp = os.path.join(WORK_DIR, f"_vout_{tag}.json")
    with open(inp, 'w') as f:
        json.dump({'programs': programs, 'train': train,
                   'test': test, 'timeout': prog_timeout}, f)
    total = 10 + prog_timeout * len(programs) + 5
    try:
        subprocess.run([sys.executable, RUNTIME_PATH, inp, outp],
                       timeout=total,
                       stdout=subprocess.DEVNULL,
                       stderr=subprocess.DEVNULL)
    except subprocess.TimeoutExpired:
        return [{'status': 'timeout', 'pairs_passed': 0,
                 'error': 'verifier batch timeout'}] * len(programs)
    finally:
        try: os.remove(inp)
        except OSError: pass
    if not os.path.exists(outp):
        return [{'status': 'error', 'pairs_passed': 0,
                 'error': 'verifier crashed'}] * len(programs)
    with open(outp) as f:
        res = json.load(f)['results']
    try: os.remove(outp)
    except OSError: pass
    return res

# ----------------------------------------------------------------------------
# Per-task solve (runs inside a GPU worker)
# ----------------------------------------------------------------------------
def gen_batch(model, tok, prompt, k, temp, max_new):
    import torch
    msgs = [{'role': 'system', 'content': SYSTEM_MSG},
            {'role': 'user',   'content': prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False,
                                   add_generation_prompt=True)
    enc = tok(text, return_tensors='pt').to(model.device)
    try:
        with torch.inference_mode():
            out = model.generate(**enc,
                                 do_sample=True,
                                 temperature=max(temp, 1e-3),
                                 top_p=0.95,
                                 num_return_sequences=k,
                                 max_new_tokens=max_new,
                                 pad_token_id=tok.eos_token_id)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if k > 2:
            return gen_batch(model, tok, prompt, k // 2, temp, max_new)
        raise
    L = enc['input_ids'].shape[1]
    return [tok.decode(o[L:], skip_special_tokens=True) for o in out]

def vote_outputs(verified, n_test):
    # verified: list of (src, result) ; result['test_outputs'] aligned by index
    outs = []
    for t in range(n_test):
        buckets = {}
        for src, res in verified:
            key = json.dumps(res['test_outputs'][t])
            buckets.setdefault(key, {'votes': 0, 'srcs': []})
            buckets[key]['votes'] += 1
            buckets[key]['srcs'].append(src)
        ranked = sorted(buckets.items(),
                        key=lambda kv: (-kv[1]['votes'],
                                        min(len(s) for s in kv[1]['srcs'])))
        top_key, top = ranked[0]
        alt = json.loads(ranked[1][0]) if len(ranked) > 1 else None
        outs.append({'attempt': json.loads(top_key),
                     'votes': top['votes'],
                     'alt': alt,
                     'program': min(top['srcs'], key=len)})
    return outs

def solve_task(tid, task, cfg, deadline_ts, rank, log):
    t0 = time.time()
    task_end = min(t0 + cfg['per_task_cap'], deadline_ts)
    rec = {'verified': False, 'n_verified': 0, 'outputs': [],
           'best_partial': 0, 'n_pairs': len(task['train']),
           'n_cand': 0, 'elapsed': 0.0}
    prompt = build_prompt(task)
    if len(prompt) > cfg['max_prompt_chars']:
        rec['skip'] = 'prompt_too_long(%d)' % len(prompt)
        log(f"[W{rank}] {tid}  SKIP prompt too long ({len(prompt)} chars)")
        return rec
    model, tok = cfg['_model'], cfg['_tok']
    test_inputs = [t['input'] for t in task['test']]

    seen, all_codes, all_res = set(), [], []
    def run_round(k, temp, repair=None):
        texts = gen_batch(model, tok, build_prompt(task, repair),
                          k, temp, cfg['max_new_tokens'])
        codes = []
        for txt in texts:
            c = extract_code(txt)
            h = code_key(c)
            if c and h not in seen:
                seen.add(h); codes.append(c)
        if not codes: return
        res = run_verifier(codes, task['train'], test_inputs,
                           cfg['prog_timeout'], f"{rank}_{tid}")
        all_codes.extend(codes); all_res.extend(res)

    run_round(cfg['k1'], cfg['temp1'])
    verified = [(c, r) for c, r in zip(all_codes, all_res)
                if r['status'] == 'verified']
    if not verified and time.time() < task_end - 30:
        partials = [(c, r) for c, r in zip(all_codes, all_res)
                    if r['status'] == 'partial' and r['pairs_passed'] >= 1]
        if partials:
            c, r = max(partials, key=lambda cr: (cr[1]['pairs_passed'],
                                                 -len(cr[0])))
            rep = {'code': c[:4000],
                   'pairs_passed': r['pairs_passed'],
                   'n_pairs': r['n_pairs'],
                   'fail_idx': r.get('idx', 0),
                   'diff': repair_diff(task, r)}
            run_round(cfg['k2'], cfg['temp2'], repair=rep)
            verified = [(c2, r2) for c2, r2 in zip(all_codes, all_res)
                        if r2['status'] == 'verified']

    rec['n_cand'] = len(all_codes)
    rec['best_partial'] = max([r.get('pairs_passed', 0) for r in all_res],
                              default=0)
    if verified:
        rec['verified']   = True
        rec['n_verified'] = len(verified)
        rec['outputs']    = vote_outputs(verified, len(test_inputs))
        rec['program']    = rec['outputs'][0]['program']
    rec['elapsed'] = round(time.time() - t0, 1)
    tag = (f"VERIFIED {rec['n_verified']} prog(s)" if rec['verified']
           else f"no verified (best {rec['best_partial']}/{rec['n_pairs']})")
    log(f"[W{rank}] {tid}  {tag}  cand {rec['n_cand']}  {rec['elapsed']}s")
    return rec

# ----------------------------------------------------------------------------
# GPU worker process (heavy imports live here -> spawn stays cheap)
# ----------------------------------------------------------------------------
def worker_main(rank, gpu_id, items, cfg, deadline_ts, partial_path):
    os.environ['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
    def log(msg): print(msg, flush=True)
    try:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        torch.manual_seed(cfg['seed'] + rank)
        random.seed(cfg['seed'] + rank)
        log(f"[W{rank}] loading {cfg['model_path']} on gpu {gpu_id} ...")
        tok = AutoTokenizer.from_pretrained(cfg['model_path'])
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            cfg['model_path'], torch_dtype=torch.bfloat16
        ).to('cuda').eval()
        cfg = dict(cfg); cfg['_model'] = model; cfg['_tok'] = tok
        log(f"[W{rank}] ready, {len(items)} tasks")
    except Exception as e:
        with open(partial_path, 'a') as f:
            f.write(json.dumps({'tid': '__worker_error__',
                                'error': repr(e)[:500]}) + "\n")
        print(f"[W{rank}] FATAL during model load: {e!r}", flush=True)
        return
    with open(partial_path, 'a') as f:
        for tid, task in items:
            if time.time() > deadline_ts - 25:
                log(f"[W{rank}] deadline reached, stopping")
                break
            try:
                rec = solve_task(tid, task, cfg, deadline_ts, rank, log)
            except Exception as e:
                rec = {'verified': False, 'error': repr(e)[:300]}
                log(f"[W{rank}] {tid}  ERROR {e!r}")
            f.write(json.dumps({'tid': tid, **{k: v for k, v in rec.items()
                                               if not k.startswith('_')}}) + "\n")
            f.flush()

# ----------------------------------------------------------------------------
# Data / model resolution
# ----------------------------------------------------------------------------
def resolve_data_dir():
    for d in DATA_DIR_CANDIDATES:
        if os.path.exists(os.path.join(d, "arc-agi_evaluation_challenges.json")) \
           or os.path.exists(os.path.join(d, "arc-agi_test_challenges.json")):
            return d
    return DATA_DIR_CANDIDATES[0]

def load_challenges():
    data_dir = resolve_data_dir()
    rerun  = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
    test_p = os.path.join(data_dir, "arc-agi_test_challenges.json")
    eval_p = os.path.join(data_dir, "arc-agi_evaluation_challenges.json")
    sol_p  = os.path.join(data_dir, "arc-agi_evaluation_solutions.json")
    if rerun or (os.path.exists(test_p) and not os.path.exists(eval_p)):
        with open(test_p) as f:
            return json.load(f), None, 'SUBMIT', data_dir
    with open(eval_p) as f:
        ch = json.load(f)
    sols = None
    if os.path.exists(sol_p):
        with open(sol_p) as f:
            sols = json.load(f)
    return ch, sols, 'LOCAL_EVAL', data_dir

def resolve_model_path(override=None):
    cands = ([override] if override else []) + MODEL_PATH_CANDIDATES
    for p in cands:
        if p and os.path.exists(os.path.join(p, "config.json")):
            return p
    root = "/kaggle/input"
    if os.path.isdir(root):
        hits = []
        for dirpath, _, files in os.walk(root):
            if dirpath.count(os.sep) - root.count(os.sep) > 7:
                continue
            if "config.json" in files and (
                "tokenizer.json" in files or "tokenizer_config.json" in files):
                low = dirpath.lower()
                if "coder" in low or "qwen" in low:
                    hits.append((0 if "7b" in low else 1, dirpath))
        if hits:
            return sorted(hits)[0][1]
    return None

def est_cost(task):
    s = 0
    for p in task['train']:
        s += len(p['input']) * len(p['input'][0])
        s += len(p['output']) * len(p['output'][0])
    for t in task['test']:
        s += len(t['input']) * len(t['input'][0])
    return s

# ----------------------------------------------------------------------------
# Selftest: proves the sandbox (and the exec fix) with zero GPU/model
# ----------------------------------------------------------------------------
def selftest():
    print("== Leg C sandbox selftest (no GPU, no model) ==")
    train = [
        {'input':  [[0,0,0],[0,3,0],[0,0,0]],
         'output': [[0,0,0],[0,2,0],[0,0,0]]},
        {'input':  [[5,0],[0,0]],
         'output': [[2,0],[0,0]]},
    ]
    test = [[[0,7,0],[0,0,0],[0,0,0]]]
    cases = [
        ("dsl_reachable (old two-dict exec bug regression)",
         "def transform(grid):\n"
         "    objs = get_objects(grid)\n"
         "    return paint_objects(grid, objs, 2)\n", 'verified'),
        ("while loop must be rejected",
         "def transform(grid):\n"
         "    while True:\n        pass\n    return grid\n", 'rejected'),
        ("import must be rejected",
         "import os\ndef transform(grid):\n    return grid\n", 'rejected'),
        ("missing transform must be rejected",
         "def solve(grid):\n    return grid\n", 'rejected'),
        ("runaway loop must time out",
         "def transform(grid):\n"
         "    x = 0\n"
         "    for i in range(10**9):\n        x += i\n"
         "    return grid\n", 'timeout'),
        ("wrong logic must be partial",
         "def transform(grid):\n    return replace_color(grid, 3, 4)\n",
         'partial'),
        ("dunder escape must be rejected",
         "def transform(grid):\n"
         "    f = (1).__class__\n    return grid\n", 'rejected'),
    ]
    programs = [src for _, src, _ in cases]
    res = run_verifier(programs, train, test, 3.0, "selftest")
    ok = True
    for (name, _, want), r in zip(cases, res):
        got = r['status']
        mark = "PASS" if got == want else "FAIL"
        if got != want: ok = False
        print(f"  [{mark}] {name:48s} -> {got} "
              f"({r.get('error','')[:60]})")
    v = res[0]
    if v['status'] == 'verified':
        pred = v['test_outputs'][0]
        want = [[0,2,0],[0,0,0],[0,0,0]]
        mark = "PASS" if pred == want else "FAIL"
        if pred != want: ok = False
        print(f"  [{mark}] verified program predicts test correctly -> {pred}")
    names = re.findall(r"([a-z_][a-z0-9_]*)\(", CHEAT_SHEET)
    import ast as _ast
    tree = _ast.parse(RUNTIME_CODE)
    defined = {n.name for n in _ast.walk(tree)
               if isinstance(n, _ast.FunctionDef)}
    missing = [n for n in set(names) - {'f', 'color_fn', 'transform'}
               if n not in defined]
    mark = "PASS" if not missing else "FAIL"
    if missing: ok = False
    print(f"  [{mark}] cheat sheet <-> runtime consistency "
          f"(missing: {missing})")
    print("== selftest", "OK ==" if ok else "FAILED ==")
    return 0 if ok else 1

# ----------------------------------------------------------------------------
# Reporting: input + guessed result + mark
# ----------------------------------------------------------------------------
def mark_and_report(results, challenges, sols, cfg):
    audit, n_ver, n_out, n_correct = {}, 0, 0, 0
    printed = 0
    for tid, rec in results.items():
        if tid == '__worker_error__':
            continue
        task = challenges[tid]
        entry = {'verified': rec.get('verified', False),
                 'n_verified': rec.get('n_verified', 0),
                 'best_partial': rec.get('best_partial', 0),
                 'n_pairs': rec.get('n_pairs', 0),
                 'elapsed': rec.get('elapsed', 0),
                 'tests': []}
        for t_idx, t in enumerate(task['test']):
            out = (rec['outputs'][t_idx]['attempt']
                   if rec.get('verified') and t_idx < len(rec.get('outputs', []))
                   else None)
            exp = sols[tid][t_idx] if sols and tid in sols else None
            mark = None
            if out is not None:
                n_out += 1
                if exp is not None:
                    mark = (out == exp)
                    n_correct += int(mark)
            entry['tests'].append({'input': t['input'],
                                   'guess': out,
                                   'expected': exp,
                                   'mark': mark})
        if rec.get('verified'):
            n_ver += 1
            if printed < cfg['print_grids']:
                printed += 1
                for t_idx, tr in enumerate(entry['tests']):
                    sym = ('?' if tr['mark'] is None
                           else ('O' if tr['mark'] else 'X'))
                    print(f"\n--- {tid} test#{t_idx}  mark={sym} "
                          f"(votes {rec['outputs'][t_idx]['votes']}"
                          f"/{rec['n_verified']}) ---")
                    print("input:");  print(render_grid(tr['input']))
                    print("guess:");  print(render_grid(tr['guess']))
                    if tr['expected'] is not None:
                        print("expected:"); print(render_grid(tr['expected']))
        audit[tid] = entry
    with open(AUDIT_PATH, 'w') as f:
        json.dump(audit, f)
    print("\n================ Leg C summary ================")
    print(f" tasks attempted      : {len(audit)}")
    print(f" tasks verified       : {n_ver}")
    print(f" test outputs guessed : {n_out}")
    if sols is not None:
        pct = 100.0 * n_correct / max(n_out, 1)
        print(f" verified & correct   : {n_correct}/{n_out}  "
              f"(precision {pct:.1f}%)")
        solved = [t for t, e in audit.items()
                  if e['verified'] and all(x['mark'] for x in e['tests'])]
        print(f" fully-correct tasks  : {len(solved)} -> {solved}")
    print(f" results -> {RESULTS_PATH}")
    print(f" audit   -> {AUDIT_PATH}")
    print("===============================================")

def write_outputs(results, challenges):
    with open(RESULTS_PATH, 'w') as f:
        json.dump(results, f)
    # standalone submission (smoke/debug only; merge path is the real one)
    sub = {}
    for tid, task in challenges.items():
        rec = results.get(tid, {})
        entries = []
        for t_idx, t in enumerate(task['test']):
            if rec.get('verified') and t_idx < len(rec.get('outputs', [])):
                o = rec['outputs'][t_idx]
                a1 = o['attempt']
                a2 = o['alt'] if o.get('alt') else a1
            else:
                a1 = a2 = t['input']
            entries.append({'attempt_1': a1, 'attempt_2': a2})
        sub[tid] = entries
    with open(SUB_PATH, 'w') as f:
        json.dump(sub, f)

# ----------------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--budget-h',     type=float, default=DEFAULTS['budget_h'])
    ap.add_argument('--per-task-cap', type=float, default=DEFAULTS['per_task_cap'])
    ap.add_argument('--k1',   type=int, default=DEFAULTS['k1'])
    ap.add_argument('--k2',   type=int, default=DEFAULTS['k2'])
    ap.add_argument('--workers', type=int, default=0)
    ap.add_argument('--model-path', type=str, default=None)
    ap.add_argument('--smoke', type=int, default=0)
    ap.add_argument('--max-tasks', type=int, default=0)
    ap.add_argument('--print-grids', type=int, default=DEFAULTS['print_grids'])
    ap.add_argument('--end-ts', type=float, default=0.0,
                    help='absolute unix ts; deadline = min(now+budget, end-ts)')
    ap.add_argument('--selftest', action='store_true')
    args = ap.parse_args()

    os.makedirs(WORK_DIR, exist_ok=True)
    with open(RUNTIME_PATH, 'w') as f:
        f.write(RUNTIME_CODE)

    if args.selftest:
        sys.exit(selftest())

    challenges, sols, mode, data_dir = load_challenges()
    print(f"[LegC] mode={mode}  data={data_dir}  tasks={len(challenges)}")

    model_path = resolve_model_path(args.model_path)
    if model_path is None:
        print("[LegC] WARNING: no coder model found under /kaggle/input.")
        print("       Attach Qwen2.5-Coder-7B-Instruct, or pass --model-path.")
        print("       Writing empty results so the merge step is a no-op.")
        write_outputs({}, challenges)
        mark_and_report({}, challenges, sols,
                        {'print_grids': 0})
        return
    print(f"[LegC] model: {model_path}")

    cfg = dict(DEFAULTS)
    cfg.update(per_task_cap=args.per_task_cap, k1=args.k1, k2=args.k2,
               print_grids=args.print_grids, model_path=model_path)

    items = sorted(challenges.items(), key=lambda kv: est_cost(kv[1]))
    if args.smoke:
        items = items[:args.smoke]
        cfg['print_grids'] = max(cfg['print_grids'], args.smoke)
    elif args.max_tasks:
        items = items[:args.max_tasks]

    try:
        n_gpu = int(subprocess.run(
            ['nvidia-smi', '-L'], capture_output=True, text=True
        ).stdout.strip().count('GPU '))
    except Exception:
        n_gpu = 1
    n_workers = args.workers or max(1, min(4, n_gpu))
    shards = [items[i::n_workers] for i in range(n_workers)]

    deadline_ts = time.time() + args.budget_h * 3600
    if args.end_ts > 0:
        deadline_ts = min(deadline_ts, args.end_ts)
    remain = deadline_ts - time.time()
    if remain < 300:
        print(f"[LegC] budget too small ({remain:.0f}s) -- writing empty "
              "results and exiting so the base pipeline keeps all its time.")
        write_outputs({}, challenges)
        mark_and_report({}, challenges, sols, {'print_grids': 0})
        return
    print(f"[LegC] {len(items)} tasks, {n_workers} worker(s), "
          f"budget {remain/3600:.2f} h")

    for r in range(n_workers):
        p = PARTIAL_TPL.format(rank=r)
        if os.path.exists(p): os.remove(p)

    ctx = mp.get_context('spawn')
    procs = []
    for r in range(n_workers):
        p = ctx.Process(target=worker_main,
                        args=(r, r % max(n_gpu, 1), shards[r], cfg,
                              deadline_ts, PARTIAL_TPL.format(rank=r)))
        p.start(); procs.append(p)
    hard_stop = deadline_ts + 600
    for p in procs:
        p.join(max(1.0, hard_stop - time.time()))
    for p in procs:
        if p.is_alive():
            p.terminate(); p.join()

    results = {}
    for r in range(n_workers):
        path = PARTIAL_TPL.format(rank=r)
        if not os.path.exists(path): continue
        with open(path) as f:
            for line in f:
                line = line.strip()
                if not line: continue
                try:
                    rec = json.loads(line)
                    results[rec.pop('tid')] = rec
                except Exception:
                    continue

    write_outputs(results, challenges)
    mark_and_report(results, challenges, sols, cfg)

if __name__ == '__main__':
    main()


Writing arc_induction_v2.py


In [10]:
# ============ Leg C sandbox selftest (CPU, ~30s) ============
# 9/9 PASS expected: DSL sandbox, AST enforcement, per-program timeouts.
!python arc_induction_v2.py --selftest


== Leg C sandbox selftest (no GPU, no model) ==
  [PASS] dsl_reachable (old two-dict exec bug regression) -> verified ()
  [PASS] while loop must be rejected                      -> rejected (banned construct: While)
  [PASS] import must be rejected                          -> rejected (banned construct: Import)
  [PASS] missing transform must be rejected               -> rejected (missing def transform(grid))
  [PASS] runaway loop must time out                       -> timeout (exceeded 3.0s)
  [PASS] wrong logic must be partial                      -> partial ()
  [PASS] dunder escape must be rejected                   -> rejected (banned dunder attribute: __class__)
  [PASS] verified program predicts test correctly -> [[0, 2, 0], [0, 0, 0], [0, 0, 0]]
  [PASS] cheat sheet <-> runtime consistency (missing: [])
== selftest OK ==


In [11]:
# ===================== Leg C launch (BEFORE base pipeline) =====================
# Verified-induction pre-pass (Qwen2.5-Coder-7B-Instruct, 4 x L4 workers).
# Base is near time-saturated on the hidden set, so Leg C wall time is NOT free:
# keep LEGC_BUDGET_H small; the starter-level skip refunds verified tasks.
# arc_induction_v2.py exits safely (empty results) if the budget is < 5 min,
# if the coder model is not attached, or on any internal failure -> the merge
# becomes a no-op and the base score is untouched.
LEGC_BUDGET_H  = 0.0   # 0 disables Leg C (safe no-op)
BASE_RESERVE_H = 9.8   # wall hours guaranteed to remain for the base pipeline

LEGC_END_TS = f"{global_end_time - BASE_RESERVE_H * 3600:.0f}"
LEGC_BUDGET = f"{LEGC_BUDGET_H:.3f}"

!python arc_induction_v2.py --budget-h {LEGC_BUDGET} --end-ts {LEGC_END_TS}


[LegC] mode=LOCAL_EVAL  data=/kaggle/working/arcdata  tasks=120
[LegC] model: /kaggle/input/models/mikedan7/qwen3-8-27b-fp8-official/pytorch/hf-fp8/1
[LegC] budget too small (-11910s) -- writing empty results and exiting so the base pipeline keeps all its time.

================ Leg C summary ================
 tasks attempted      : 0
 tasks verified       : 0
 test outputs guessed : 0
 verified & correct   : 0/0  (precision 0.0%)
 fully-correct tasks  : 0 -> []
 results -> /kaggle/working/induction_results.json
 audit   -> /kaggle/working/induction_audit.json


In [12]:
!UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 python starter.py --end-time {global_end_time}

[LegC] skipping 0 verified task(s) in the base queue
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
MODEL_PATH /kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1
[Rank 0] start!
==((====))==  Unsloth 2025.9.7: Fast Qwen3 patching. Transformers: 4.55.4.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading checkpoint shards:  50%|█████████         | 1/2 [00:35<00:35, 35.87s/it]🦥 Unsloth Zoo will now patch everything to make training faster!
MODEL_PATH /kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1
[Rank 1] start!
=

In [13]:
import traceback
try:
    exec(compile('import os\nimport json\nfrom arc_loader import ArcDataset\nfrom arc_decoder import ArcDecoder\n\nrerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")\n\nif rerun_mode:\n    data = ArcDataset.from_file("/kaggle/working/arcdata/arc-agi_test_challenges.json")\nelse:\n    data = ArcDataset.from_file("/kaggle/working/arcdata/arc-agi_evaluation_challenges.json")\n    data = data.load_replies("/kaggle/working/arcdata/arc-agi_evaluation_solutions.json")\n\ndecoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)\n\ndecoder.load_decoded_results("/kaggle/inference_outputs")\n\nsubmission = data.get_submission(decoder.run_selection_algo())\n\nwith open("submission.json", "w") as f:\n    json.dump(submission, f)\n\nif not rerun_mode:\n    decoder.benchmark_selection_algos()\n    with open("submission.json", "r") as f:\n        reload_submission = json.load(f)\n    print("*** Reload score:", data.validate_submission(reload_submission))', 'added_cell', 'exec'), globals())
except BaseException as _e:
    print('ADDED CELL FAILED (NVARC submission kept):', repr(_e)); traceback.print_exc()


*** Load solutions from '/kaggle/working/arcdata/arc-agi_evaluation_solutions.json'...
*** Generating submission for 5 outputs...
*** Benchmark selection algorithms...
ALL_CORRECT: 0.00072 -  1.78473 16x16 [36a08778_0.transpose.permute4950812673.ex024513.out0]
ALL_CORRECT: 0.00055 -  1.78473 16x16 [36a08778_0.permute4150723698.ex504312.out0]
ALL_CORRECT: 0.00280 -  1.78473 16x16 [36a08778_0.transpose.rot90.rot90.rot90.permute4965803712.ex250413.out0]
ALL_CORRECT: 0.25327 -  1.78473 16x16 [36a08778_0.transpose.rot90.permute1654278039.ex423501.out0]
ALL_CORRECT: 0.00000 -  0.00001 30x30 [981571dc_0.transpose.rot90.rot90.rot90.permute7690351428.ex203.out0]
ALL_CORRECT: 0.00000 -  0.00001 30x30 [981571dc_0.transpose.rot90.permute5980213764.ex302.out0]
ALL_CORRECT: 0.00002 -  0.00001 30x30 [981571dc_0.transpose.rot90.rot90.permute5436201789.ex310.out0]
ALL_CORRECT: 0.00000 -  0.00001 30x30 [981571dc_0.rot90.rot90.rot90.permute1654278039.ex120.out0]
ALL_CORRECT: 0.00000 -  0.00001 30x30 [981

In [14]:
import traceback
try:
    exec(compile('# ===================== Leg C merge (run AFTER the submission cell) =====================\n# Monotonic union: train-verified induction outputs pre-empt attempt_1, the\n# base candidate is demoted to attempt_2 (cross-family decorrelation).\n# Tasks without a verified program are left untouched.\nimport os, json\n\nSUB = "submission.json"\nIND = "/kaggle/working/induction_results.json"\n\nwith open(SUB) as f:\n    sub = json.load(f)\nind = {}\nif os.path.exists(IND):\n    with open(IND) as f:\n        ind = json.load(f)\n\nrerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")\n\ndef local_score(s):\n    try:\n        return data.validate_submission(s)   # `data` from the submission cell\n    except Exception:\n        return None\n\nbefore = None if rerun_mode else local_score(sub)\n\npromoted = created = 0\nfor tid, rec in ind.items():\n    if not isinstance(rec, dict) or not rec.get("verified"):\n        continue\n    for ti, o in enumerate(rec.get("outputs", [])):\n        g = o.get("attempt")\n        if not g:\n            continue\n        alt = o.get("alt")\n        if tid not in sub:\n            sub[tid] = []\n        while len(sub[tid]) <= ti:\n            sub[tid].append({"attempt_1": [[0]], "attempt_2": [[0]]})\n            created += 1\n        e = sub[tid][ti]\n        if e["attempt_1"] != g:\n            demoted = e["attempt_1"]\n            e["attempt_2"] = demoted if demoted != [[0]] else (alt or demoted)\n            e["attempt_1"] = g\n            promoted += 1\n        elif alt and e.get("attempt_2") == [[0]]:\n            e["attempt_2"] = alt\n\nwith open(SUB, "w") as f:\n    json.dump(sub, f)\n\nafter = None if rerun_mode else local_score(sub)\nprint(f"[LegC merge] promoted={promoted} created={created}")\nif before is not None and after is not None:\n    print(f"[LegC merge] local score: base {before:.2f} -> merged {after:.2f} (unit = tasks)")\n', 'added_cell', 'exec'), globals())
except BaseException as _e:
    print('ADDED CELL FAILED (NVARC submission kept):', repr(_e)); traceback.print_exc()


[LegC merge] promoted=0 created=0
[LegC merge] local score: base 3.00 -> merged 3.00 (unit = tasks)


In [15]:
import traceback
try:
    exec(compile('# === diagnostics after NVARC + backup of the NVARC-only submission ===\nimport os, shutil, subprocess, time, json\nprint(f"elapsed {(time.time() - _T0) / 3600:.2f} h since start", flush=True)\nsubprocess.run("df -h /kaggle/working /kaggle/tmp / | head; free -g; nvidia-smi --query-gpu=memory.used --format=csv; ls /kaggle/inference_outputs | wc -l", shell=True)\nshutil.copy("submission.json", "/kaggle/working/submission_nvarc.json")\n# per test output: number of NVARC samples behind the kgmon #2 candidate (merge_sim: replace #2 only if < 8 -> +0.4..1.1)\ntry:\n    from arc_decoder import score_kgmon, hashable\n    _c2 = {}\n    for _bk, _v in decoder.decoded_results.items():\n        _groups = {}\n        for _g in _v.values():\n            _groups.setdefault(hashable(_g["solution"]), []).append(_g)\n        _ranked = score_kgmon(_v)\n        _c2[_bk] = len(_groups[hashable(_ranked[1])]) if len(_ranked) > 1 else 0\n    json.dump(_c2, open("/kaggle/working/nvarc_count2.json", "w"))\n    print("NVARC #2 counts:", len(_c2), "outputs; >=8:", sum(c >= 8 for c in _c2.values()))\nexcept Exception as _e:\n    print("count2 skipped:", repr(_e))\nprint("NVARC submission backed up:", len(json.load(open("submission.json"))), "tasks", flush=True)\n', 'added_cell', 'exec'), globals())
except BaseException as _e:
    print('ADDED CELL FAILED (NVARC submission kept):', repr(_e)); traceback.print_exc()


elapsed 0.28 h since start
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G  4.0M   20G   1% /kaggle/working
overlay         8.0T  7.0T  1.1T  88% /
overlay         8.0T  7.0T  1.1T  88% /
               total        used        free      shared  buff/cache   available
Mem:             188           1         145           0          41         185
Swap:              0           0           0
memory.used [MiB]
0 MiB
0 MiB
0 MiB
0 MiB
37
NVARC #2 counts: 5 outputs; >=8: 0
NVARC submission backed up: 120 tasks


In [16]:
import traceback
try:
    exec(compile('# === Qwen phase 1: launch vLLM (TP=4 on L4) — any failure keeps the NVARC submission ===\nimport os, subprocess, time, json, urllib.request\nQWEN_OK = False\ndef _log(f): return open(f, errors="ignore").read() if os.path.exists(f) else ""\nt0 = time.time()\nwhile not any(k in _log("/kaggle/working/venv_setup.log") for k in ("VENV_DONE", "VENV_FAILED")) and time.time() - t0 < 1800:\n    time.sleep(10)\nprint(_log("/kaggle/working/venv_setup.log")[-1500:])\nif "VENV_DONE" in _log("/kaggle/working/venv_setup.log") and time.time() < QWEN_DEADLINE - 1800:\n    subprocess.run("pkill -f starter.py; sleep 5; nvidia-smi --query-gpu=memory.used --format=csv", shell=True)\n    env = dict(os.environ)\n    env.update(PYTORCH_ALLOC_CONF="expandable_segments:True", VLLM_FORCE_ATTN_BACKEND="TRITON_ATTN",\n               PATH=VENV + "/bin:" + env.get("PATH", ""), PYTHONPATH="")\n    if os.path.exists("/usr/local/nvidia/lib64/libcuda.so"):\n        env["LD_LIBRARY_PATH"] = "/usr/local/nvidia/lib64:" + env.get("LD_LIBRARY_PATH", "")\n        env["LD_PRELOAD"] = "/usr/local/nvidia/lib64/libcuda.so"\n    cmd = [VPY, "-m", "vllm.entrypoints.openai.api_server", "--model", QWEN_MODEL,\n           "--served-model-name", "qwen3.8-27b", "--port", "8000", "--tensor-parallel-size", "4",\n           "--gpu-memory-utilization", "0.94", "--max-model-len", "131072", "--max-num-seqs", str(QWEN_WORKERS),\n           "--trust-remote-code", "--kv-cache-dtype", "fp8", "--attention-backend", "TRITON_ATTN",\n           "--no-async-scheduling", "--reasoning-parser", "qwen3",\n           *(["--default-chat-template-kwargs", json.dumps({"reasoning_effort": QWEN_EFFORT})] if QWEN_EFFORT else []),\n           "--speculative-config", json.dumps({"method": "mtp", "num_speculative_tokens": 2})]\n    t1 = time.time()\n    for extra in [*QWEN_EXTRA_TRIES, []]:  # newer flags first; fall back to the validated v4 command if vLLM rejects them\n        if QWEN_OK or time.time() > QWEN_DEADLINE - 1800:\n            break\n        print("vLLM extra args:", extra, flush=True)\n        srv = subprocess.Popen(cmd + extra, stdout=open("/kaggle/working/vllm.log", "a"), stderr=subprocess.STDOUT,\n                               env=env, start_new_session=True)\n        t2 = time.time()\n        while time.time() - t2 < 1500:\n            try:\n                urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=3); QWEN_OK = True; break\n            except Exception:\n                if srv.poll() is not None: break\n                time.sleep(10)\n        if not QWEN_OK:\n            try:\n                os.killpg(srv.pid, 9)\n            except Exception:\n                pass\n            subprocess.run(["pkill", "-9", "-f", "vllmenv"])\n            for _ in range(30):  # wait until the dead try\'s workers have released the GPUs\n                time.sleep(10)\n                used = subprocess.run("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits", shell=True,\n                                      capture_output=True, text=True).stdout.split()\n                if used and max(int(x) for x in used) < 1500:\n                    break\n            print("GPU memory after failed try (MiB):", used, flush=True)\n        if QWEN_OK or not extra:\n            break\n    print("vLLM up" if QWEN_OK else "vLLM FAILED", f"{time.time()-t1:.0f}s", "args:", extra if QWEN_OK else None)\n    if not QWEN_OK:\n        print("\\n".join(l for l in _log("/kaggle/working/vllm.log").splitlines()\n                        if any(k in l for k in ("Error", "error", "CUDA", "Traceback")))[-3000:])\nelse:\n    print("skip Qwen: venv not ready or not enough time left")\n', 'added_cell', 'exec'), globals())
except BaseException as _e:
    print('ADDED CELL FAILED (NVARC submission kept):', repr(_e)); traceback.print_exc()


+ unset PYTHONPATH PYTHONHOME PYTHONSTARTUP
+ export PYTHONNOUSERSITE=1
+ PYTHONNOUSERSITE=1
+ mkdir -p /kaggle/tmp/py312
+ tar -xzf /kaggle/input/datasets/yukuo14/py312-standalone-linux/cpython312_install_only.tgzbin -C /kaggle/tmp/py312
+ /kaggle/tmp/py312/python/bin/python3.12 -m venv /kaggle/tmp/vllmenv
+ /kaggle/tmp/vllmenv/bin/pip install -q --no-index --find-links /kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3 -r /kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3/requirements.lock
+ /kaggle/tmp/vllmenv/bin/pip install -q --no-index --find-links /kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3 ninja
++ ls /kaggle/tmp/vllmenv/lib/python3.12/site-packages/vllm/model_executor/layers/attention/attention.py
+ for f in $(ls /kaggle/tmp/vllmenv/lib/python3.12/site-packages/vllm/model_executor/layers/attention/attention.py 2>/dev/null)
+ grep -q 'extra_impl_args.pop('\''layer_idx'\'', None)' /kaggle/tmp/vllmenv/lib/python3.12/si

In [17]:
import traceback
try:
    exec(compile('# === Qwen phase 2: direct answers, then merge into submission.json ===\nimport json, os, time, importlib.util\nif QWEN_OK:\n    spec = importlib.util.spec_from_file_location("qs", "/kaggle/working/qwen_solver.py")\n    qs = importlib.util.module_from_spec(spec); spec.loader.exec_module(qs)\n    rerun = os.getenv("KAGGLE_IS_COMPETITION_RERUN")\n    os.environ["QWEN_TOKS_PER_S"] = str(QWEN_TPS)  # per-request speed used to cap each generation to the time left\n    ch = json.load(open("/kaggle/working/arcdata/" + ("arc-agi_test_challenges.json" if rerun else "arc-agi_evaluation_challenges.json")))\n    keys = sorted(ch) if rerun else sorted(ch, key=(lambda k: qs.task_size(ch[k])) if os.environ.get("QWEN_PROGRAM") else None)[:QWEN_COMMIT_TASKS]\n    OUT = "/kaggle/working/qwen_out"\n    dl = QWEN_DEADLINE if rerun else min(QWEN_DEADLINE, time.time() + QWEN_COMMIT_MIN * 60)\n    t0 = time.time()\n    _sub0 = json.load(open("submission.json"))\n    first = {k for k in keys if k in _sub0 and all(t["attempt_1"] == [[0]] for t in _sub0[k])}  # NVARC left these empty\n    print("tasks without an NVARC answer (Qwen takes them first):", len(first))\n    qs.solve_all({k: ch[k] for k in keys}, OUT, dl, workers=QWEN_WORKERS, solver=qs.solve_direct, max_tokens=QWEN_MAX_TOKENS, first=first)\n    recs = {f[:-5]: json.load(open(os.path.join(OUT, f))) for f in os.listdir(OUT) if f.endswith(".json")}\n    print(f"qwen: {len(recs)}/{len(keys)} tasks, answered {sum(any(g is not None for g in r[\'tests\']) for r in recs.values())}, {(time.time()-t0)/60:.0f} min")\n    sub = json.load(open("submission.json")); n = 0; kept = 0; promoted = 0\n    NV2_COUNT = json.load(open("/kaggle/working/nvarc_count2.json")) if os.path.exists("/kaggle/working/nvarc_count2.json") else {}\n    for k, r in recs.items():\n        for i, g in enumerate(r["tests"]):\n            if g is None or k not in sub or i >= len(sub[k]): continue\n            if r.get("verified"):                 # v13: program reproduces every train pair -> #1, NVARC #1 -> #2\n                if sub[k][i]["attempt_1"] != g:\n                    sub[k][i]["attempt_2"] = sub[k][i]["attempt_1"]; sub[k][i]["attempt_1"] = g; promoted += 1\n                continue\n            if sub[k][i]["attempt_1"] != g:\n                c2 = NV2_COUNT.get(f"{k}_{i}", 0)  # how many NVARC samples produced its current attempt_2\n                natural = r.get("finish") == "stop" and not r.get("forced")  # G4: ~77% right vs NVARC #2 at >=3 votes 13%\n                if QWEN_MERGE_MAXCOUNT and c2 >= QWEN_MERGE_MAXCOUNT and not (QWEN_STOP_OVERRIDE and natural):\n                    kept += 1; continue           # NVARC is fairly sure of its #2: keep it\n                if r.get("forced_tokens") and c2 > 0:\n                    kept += 1; continue           # answer forced out of a truncated trace (~6% right): only fills an empty #2\n                sub[k][i]["attempt_2"] = g; n += 1\n    json.dump(sub, open("submission.json", "w"))\n    print("merged qwen answers into attempt_2:", n, "| kept confident NVARC #2:", kept,\n          "| verified tasks:", sum(bool(r.get("verified")) for r in recs.values()), "promoted to attempt_1:", promoted)\n    if not rerun:\n        sol = json.load(open("/kaggle/working/arcdata/arc-agi_evaluation_solutions.json"))\n        ok = sum(any(sub[k][i][a] == s for a in ("attempt_1", "attempt_2")) / len(sol[k]) for k in keys for i, s in enumerate(sol[k]))\n        print(f"commit check on {len(keys)} eval tasks (hybrid): {ok:.2f}")\n    toks = [r.get("completion_tokens") or 0 for r in recs.values()]\n    fins = [r.get("finish") for r in recs.values()]\n    print("tokens mean", int(sum(toks) / max(1, len(toks))), "max", max(toks or [0]), "finish", {f: fins.count(f) for f in set(fins)})\n    print("seconds per task:", sorted(round(r["seconds"]) for r in recs.values()))\n    print("v14: forced", sum(bool(r.get("forced")) for r in recs.values()), "force s", sorted(r.get("force_s") or 0 for r in recs.values() if r.get("forced")),\n          "token-id forcing", sum(bool(r.get("force_ids")) for r in recs.values()), "| repair tried", sum("repair" in r for r in recs.values()),\n          "repaired", sum(bool(r.get("repaired")) for r in recs.values()), "| suspect", sum(bool(r.get("suspect")) for r in recs.values()))\n    print("prefix cache:", [l.split("Prefix cache hit rate:")[-1].strip() for l in _log("/kaggle/working/vllm.log").splitlines() if "Prefix cache hit rate" in l][-5:])\n    print("\\n".join(l for l in _log("/kaggle/working/vllm.log").splitlines() if "throughput" in l.lower())[-2000:])\n    subprocess.run("pkill -f vllm.entrypoints", shell=True)\n', 'added_cell', 'exec'), globals())
except BaseException as _e:
    print('ADDED CELL FAILED (NVARC submission kept):', repr(_e)); traceback.print_exc()


tasks without an NVARC answer (Qwen takes them first): 24
qwen: 24/24 tasks, answered 18, 45 min
merged qwen answers into attempt_2: 14 | kept confident NVARC #2: 0 | verified tasks: 9 promoted to attempt_1: 13
commit check on 24 eval tasks (hybrid): 7.00
tokens mean 37517 max 39149 finish {None: 1, 'length': 23}
seconds per task: [2042, 2222, 2226, 2230, 2373, 2390, 2532, 2598, 2604, 2609, 2621, 2643, 2646, 2647, 2652, 2653, 2654, 2663, 2675, 2688, 2689, 2690, 2690, 2690]
v14: forced 18 force s [41.8, 42.6, 48.1, 50.2, 135.5, 144.0, 149.0, 157.6, 159.6, 174.8, 217.1, 250.6, 262.8, 276.5, 323.5, 336.3, 373.3, 380.8] token-id forcing 18 | repair tried 0 repaired 0 | suspect 0
prefix cache: ['9.0%', '9.0%', '9.0%', '9.0%', '9.0%']
 reqs, GPU KV cache usage: 25.3%, Prefix cache hit rate: 9.0%
(APIServer pid=2526) INFO 10-28 13:10:10 [metrics.py:101] SpecDecoding metrics: Mean acceptance length: 2.69, Accepted throughput: 143.10 tokens/s, Drafted throughput: 169.59 tokens/s, Accepted: 1431

In [18]:
import traceback
try:
    exec(compile('# === final schema check: never ship a broken file; fall back to the NVARC-only submission ===\nimport json, os, shutil\ndef _ok_grid(g):\n    return isinstance(g, list) and 0 < len(g) <= 30 and all(isinstance(r, list) and 0 < len(r) <= 30 and len(r) == len(g[0])\n                                                          and all(isinstance(v, int) and 0 <= v <= 9 for v in r) for r in g)\ndef _valid(path, ref):\n    try:\n        sub = json.load(open(path))\n        return all(k in sub and len(sub[k]) == len(v["test"]) and all(_ok_grid(t["attempt_1"]) and _ok_grid(t["attempt_2"]) for t in sub[k])\n                   for k, v in ref.items())\n    except Exception as e:\n        print("check error", repr(e)); return False\ntry:\n    _ref = json.load(open("/kaggle/working/arcdata/" + ("arc-agi_test_challenges.json" if os.getenv("KAGGLE_IS_COMPETITION_RERUN") else "arc-agi_evaluation_challenges.json")))\n    if not _valid("submission.json", _ref) and os.path.exists("/kaggle/working/submission_nvarc.json"):\n        print("merged submission INVALID -> restoring NVARC-only submission")\n        shutil.copy("/kaggle/working/submission_nvarc.json", "submission.json")\n    print("final submission valid:", _valid("submission.json", _ref))\nexcept Exception as e:\n    print("final check skipped:", repr(e))\n', 'added_cell', 'exec'), globals())
except BaseException as _e:
    print('ADDED CELL FAILED (NVARC submission kept):', repr(_e)); traceback.print_exc()


final submission valid: True
